# Deep Learning-Based Video Quality Assessment for 3D Reconstruction

**Predicting whether a video is suitable for 3D reconstruction — before running the expensive reconstruction.**

---

## 1. Project objectives and methodology

### 1.1 Objective

Given a video of an object or scene, predict a **reconstruction-quality score in the range 0–100** and a
binary **Good / Poor** suitability label, so that a downstream system can skip (or prioritise) an
expensive Structure-from-Motion / multi-view-stereo reconstruction.

The model **predicts suitability only**. It never produces a 3D model.

### 1.2 Why this is non-trivial

CO3D does **not** ship a universal "Good/Poor reconstruction quality" label for every sequence. So the
ground truth has to be *constructed* by actually running a reconstruction pipeline (COLMAP) over
representative sequences and measuring how well it worked. That is Phase 4 below, and it is the
single most important methodological decision in this project:

> **The label is a measured property of a real COLMAP run, not a human opinion and not a proxy.**

### 1.3 Methodology at a glance

| Phase | What happens | Key principle |
|---|---|---|
| 2 | Download a **small subset** of CO3D (1–3 categories) | Do not download 300 GB |
| 3 | Inspect structure, annotations, splits, data quality | Verify fields, never assume them |
| 4 | Run COLMAP SfM on sampled frames, harvest metrics | Ground truth = measured reconstruction |
| 5 | Combine metrics into a 0–100 score, derive Good/Poor | Documented, reproducible, justified |
| 6 | Quality checks, sequence-level split (70/15/15) | Split by **sequence**, never by frame |
| 7 | Sample N frames in temporal order → 224×224 tensors | One sequence = one training example |
| 8 | ResNet-18 spatial features + temporal motion features | **No GT camera/depth/point-cloud at inference** |
| 9 | ResNet-18 → LSTM → regression head | Frozen CNN first, then partial fine-tune |
| 10 | Train with Huber loss, early stopping, checkpointing | Simple and reproducible |
| 11 | MAE / RMSE / R² + Accuracy / F1 / ROC-AUC + baselines | Compare against trivial baselines |
| 12 | Error analysis by failure mode and category | Understand *where* it fails |
| 13 | Save model + config + threshold | Inference must not need COLMAP |
| 14 | `predict_video()` for a new video | End-to-end reusable function |
| 15 | Conclusions, limitations, next steps | Be honest about scope |

### 1.4 What each CO3D component contributes

| Component | Role in this project | Used as model input at inference? |
|---|---|---|
| `images/` | The **only** raw signal the model sees. Frames are the input video. | **Yes** |
| `masks/` | Foreground segmentation. Used to (a) evaluate object coverage, (b) optionally crop background. | **No** (GT annotation) |
| `depths/` + `depth_masks/` | Reference geometry to check a COLMAP point cloud against. | **No** (GT annotation) |
| `pointcloud.ply` | Reference surface for a completeness/coverage metric. | **No** (GT annotation) |
| `frame_annotations.jgz` | Frame→image path mapping, frame ordering, **camera intrinsics/extrinsics**. | **No** (GT annotation) |
| `sequence_annotations.jgz` | Sequence metadata: video path, duration, point-cloud quality, viewpoint quality. | **No** (GT annotation) |
| `set_lists/` | Official train/val/test frame splits. Used for reference and sanity checks. | — |

> **Leakage guardrail.** Ground-truth camera poses, depth maps, point clouds, and reconstruction metrics
> are *label-generation and evaluation* material only. Feeding any of them to the network at inference
> time would leak the answer and would defeat the purpose of the tool (skipping reconstruction). The
> inference path in Section 14 uses RGB frames and cheap computed features only.

### 1.5 A note on the scale of this project

This is a **student-scale prototype**. The realistic label budget is tens to low hundreds of COLMAP runs,
not tens of thousands. The pipeline is therefore written to be *modular and resumable*: every stage caches
its output to disk so it can be re-run and extended without recomputation. Scaling up is a matter of
running more sequences, not rewriting code.

## 2. Environment setup, dependency checks, and directory configuration

This cell checks for the dependencies the project needs and reports clearly what is missing rather than
crashing with an opaque `ImportError`. Missing optional dependencies (COLMAP, pycolmap) degrade the
label pipeline to "not available" instead of killing the whole notebook.

In [41]:
import os
import sys
import json
import gzip
import shutil
import subprocess
import warnings
import random
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ---------------------------------------------------------------------------------------------
# Global configuration — configured for Google Colab (/content/...)
# ---------------------------------------------------------------------------------------------
CONFIG = {
    # --- Dataset paths ---
    "CO3D_ROOT": Path(os.environ.get("CO3D_ROOT", "/content/co3d_data")),
    "CATEGORIES": ["apple", "hydrant"],   # 1-2 categories for fast, practical execution in Colab
    "WORK_DIR": Path("/content/work"),

    # --- Frame sampling / preprocessing ---
    "NUM_FRAMES": 16,          # frames sampled per sequence
    "IMG_SIZE": 224,           # ResNet-18 / ImageNet input resolution
    "BATCH_SIZE": 8,

    # --- Model ---
    "EMBED_DIM": 512,          # ResNet-18 pooled feature dimension
    "LSTM_HIDDEN": 256,
    "LSTM_LAYERS": 1,
    "DROPOUT": 0.2,

    # --- Training ---
    "LR_CNN_FROZEN": 1e-3,
    "LR_CNN_FINETUNE": 1e-5,
    "LR_HEAD": 1e-3,
    "WEIGHT_DECAY": 1e-4,
    "MAX_EPOCHS": 60,
    "EARLY_STOP_PATIENCE": 10,
    "SEED": 42,

    # --- Label generation (COLMAP) ---
    "COLMAP_MAX_IMAGES": 16,   # frames fed to COLMAP per sequence
    "COLMAP_MIN_MATCHES": 15,
    "COLMAP_MIN_NUM_REG": 3,

    # --- Quality score weighting (must sum to 1.0) ---
    "SCORE_WEIGHTS": {
        "registration_ratio": 0.35,   # how many frames were successfully registered
        "reproj_error":       0.25,   # geometric accuracy (lower is better)
        "n_points_density":   0.20,   # how much 3D structure was recovered
        "coverage":           0.20,   # coverage of the reference object surface
    },
    "GOOD_SCORE_THRESHOLD": 50.0,    # provisional; recalibrated on validation set in Section 6
}

SEED = CONFIG["SEED"]

# Create working directories under /content/work
WORK = CONFIG["WORK_DIR"]
DIRS = {
    "frames":   WORK / "frames",
    "colmap":   WORK / "colmap",
    "labels":   WORK / "labels",
    "features": WORK / "features",
    "models":   WORK / "models",
    "figures":  WORK / "figures",
}
for _d in DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------------------------
# Dependency check
# ---------------------------------------------------------------------------------------------
def check_dependencies() -> Dict[str, bool]:
    """Return a {package: available} map."""
    result = {}
    for mod in ["torch", "torchvision", "cv2", "sklearn", "matplotlib", "PIL", "tqdm"]:
        try:
            __import__(mod)
            result[mod] = True
        except Exception:
            result[mod] = False
    result["pycolmap"] = shutil.which("colmap") is not None
    return result

DEPS = check_dependencies()

print("=" * 72)
print("DEPENDENCY CHECK")
print("=" * 72)
for pkg, ok in DEPS.items():
    print(f"  {'[ OK ]' if ok else '[MISS]'}  {pkg}")

REQUIRED = ["torch", "torchvision", "sklearn", "matplotlib", "PIL"]
missing_required = [p for p in REQUIRED if not DEPS[p]]
if missing_required:
    print("\n" + "=" * 72)
    print("MISSING REQUIRED DEPENDENCIES: " + ", ".join(missing_required))
    print("=" * 72)
    print(f"Install with: !pip install {' '.join(missing_required)}")
else:
    print("\nAll required dependencies present.")

if not DEPS["pycolmap"]:
    print("\nNOTE: COLMAP was not found on PATH.")
    print("Install in Colab using:")
    print("  !apt-get install -y colmap")
    print("  !pip install pycolmap")

# Seed everything for reproducibility
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    if torch.cuda.is_available():
        print(f"\nCUDA available: {torch.cuda.get_device_name(0)}")
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        print("\nCUDA not available — training will use CPU (much slower).")
except Exception:
    pass

print(f"\nCO3D_ROOT : {CONFIG['CO3D_ROOT']}  (exists: {CONFIG['CO3D_ROOT'].exists()})")
print(f"WORK_DIR  : {WORK}")
print(f"Seed      : {SEED}")

DEPENDENCY CHECK
  [ OK ]  torch
  [ OK ]  torchvision
  [ OK ]  cv2
  [ OK ]  sklearn
  [ OK ]  matplotlib
  [ OK ]  PIL
  [ OK ]  tqdm
  [ OK ]  pycolmap

All required dependencies present.

CUDA available: Tesla T4

CO3D_ROOT : /content/co3d_data  (exists: False)
WORK_DIR  : /content/work
Seed      : 42


In [43]:
# 1. Create the destination directory first
!mkdir -p /content/co3d_data

# 2. Clone the repository if it isn't already present
import os
if not os.path.exists("/content/co3d_repo"):
    !git clone https://github.com/facebookresearch/co3d.git /content/co3d_repo

# 3. Install and run the download script
%cd /content/co3d_repo
!pip install -e . -q
!python ./co3d/download_dataset.py --download_folder /content/co3d_data --download_categories apple
%cd /content

/content/co3d_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for co3d (pyproject.toml) ... done
  0% 0/1 [00:00<?, ?it/s]https://dl.fbaipublicfiles.com/co3dv2_231130/category_to_subset_name_list.json
/content/co3d_data/category_to_subset_name_list.json: 0.00iB [00:00, ?iB/s]category_to_subset_name_list.json: Downloaded 472800.0%.
/content/co3d_data/category_to_subset_name_list.json: 4.62kiB [00:00, 10.5MiB/s]
/content/co3d_data/category_to_subset_name_list.json: 4.62kiB [00:00, 7.04MiB/s]
100% 1/1 [00:00<00:00,  5.52it/s]
  0% 0/7 [00:00<?, ?it/s]Downloading dataset file apple_000.zip (https://dl.fbaipublicfiles.com/co3dv2_231130/apple_000.zip) to /content/co3d_data/_in_progress/apple_000.zip.
https://dl.fbaipublicfiles.com/co3dv2_231130/apple_003.zip
/content/co3d_data/_in_progress/apple_003.zip:   0% 

## 3. Dataset download and directory configuration

CO3D is large (hundreds of GB for the full set), so this project deliberately works on a **small subset**.

### 3.1 Two supported acquisition routes

**Route A — official downloader (recommended, most reliable)**

```bash
git clone https://github.com/facebookresearch/co3d.git
cd co3d
pip install -e .
python ./co3d/download_dataset.py --download_folder $CO3D_ROOT
```

To pull only specific categories, download the per-category archives listed in
`dataset/links.json` and extract them so the layout is `CO3D_ROOT/<category>/<sequence>/...`.

**Route B — single-sequence subset (lightest, good for a first run)**

```bash
python ./co3d/download_dataset.py --download_folder $CO3D_ROOT --single_sequence_subset
```

This downloads only the sequences used by the many-view single-sequence task — a small, representative
slice. It is the fastest way to get a working end-to-end pipeline.

### 3.2 Expected on-disk layout

```
CO3D_ROOT/
├── <category>/
│   ├── <sequence_name>/
│   │   ├── images/          # frame000000.jpg ...
│   │   ├── masks/           # foreground masks
│   │   ├── depths/          # depth maps (where available)
│   │   ├── depth_masks/     # valid-depth masks
│   │   └── pointcloud.ply   # reference 3D point cloud
│   ├── set_lists/           # set_lists_<subset>.json
│   ├── eval_batches/
│   ├── frame_annotations.jgz
│   └── sequence_annotations.jgz
├── category_to_subset_name_list.json
└── set_lists/
```

> **Important gotcha (from the official README):** `frame_number` comes **only** from
> `frame_annotations.jgz` and does **not** necessarily match the numeric suffix of the image filename.
> `frame00005.jpg` may have `frame_number == 20`. Never sort or index frames by filename alone —
> always join through the frame annotations. Section 7 enforces this.

In [9]:
# ---------------------------------------------------------------------------------------------
# Dataset configuration + verification of the on-disk layout.
# ---------------------------------------------------------------------------------------------

def verify_co3d_layout(root: Path, categories: Sequence[str]) -> Dict[str, Any]:
    """Inspect the CO3D root and report what is present per category."""
    report: Dict[str, Any] = {
        "root": str(root),
        "root_exists": root.exists(),
        "categories_found": [],
        "per_category": {},
    }
    if not root.exists():
        report["error"] = (
            f"CO3D root not found at {root}. Please download the dataset first."
        )
        return report

    for cat in categories:
        cat_dir = root / cat
        entry: Dict[str, Any] = {
            "exists": cat_dir.exists(),
            "has_frame_annotations": (cat_dir / "frame_annotations.jgz").exists(),
            "has_sequence_annotations": (cat_dir / "sequence_annotations.jgz").exists(),
            "n_sequence_dirs": 0,
            "n_set_lists": 0,
        }
        if cat_dir.exists():
            seq_dirs = [d for d in cat_dir.iterdir() if d.is_dir() and (d / "images").is_dir()]
            entry["n_sequence_dirs"] = len(seq_dirs)
            sl_dir = cat_dir / "set_lists"
            if sl_dir.is_dir():
                entry["n_set_lists"] = len(list(sl_dir.glob("*.json")))
            report["categories_found"].append(cat)
        report["per_category"][cat] = entry

    report["category_to_subset_name_list"] = (root / "category_to_subset_name_list.json").exists()
    return report


layout = verify_co3d_layout(CONFIG["CO3D_ROOT"], CONFIG["CATEGORIES"])

print("=" * 72)
print("CO3D LAYOUT VERIFICATION")
print("=" * 72)
print(f"Root: {layout['root']}  (exists: {layout['root_exists']})")

if not layout["root_exists"] or not layout["categories_found"]:
    print("\n" + "!" * 72)
    print("DATASET NOT PRESENT OR CATEGORY EMPTY.")
    print("Run the download snippet above, then re-run this cell.")
    print("!" * 72)
else:
    print(f"category_to_subset_name_list.json present: {layout['category_to_subset_name_list']}")
    print()
    for cat, e in layout["per_category"].items():
        print(f"  {cat:<12} exists={str(e['exists']):<5} "
              f"seq_dirs={e['n_sequence_dirs']:<4} "
              f"frame_ann={str(e['has_frame_annotations']):<5} "
              f"seq_ann={str(e['has_sequence_annotations']):<5} "
              f"set_lists={e['n_set_lists']}")

CO3D LAYOUT VERIFICATION
Root: /content/co3d_data  (exists: False)

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
DATASET NOT PRESENT OR CATEGORY EMPTY.
Run the download snippet above, then re-run this cell.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


## 4. Dataset inspection and exploratory data analysis

This section answers the eight questions the project brief asks, **from the actual files on disk**:

1. Number of categories and sequences
2. Number of frames per sequence
3. Image dimensions and formats
4. Missing or corrupted files
5. Availability of masks, depths, camera annotations, point clouds
6. Distribution of sequence lengths
7. Example frames visualised in chronological order
8. Available train / validation / test splits

### 4.1 How the annotations are read

The `.jgz` files are **gzipped JSON**, so they can be read without installing the `co3d` package:

```python
with gzip.open(path, "rt", encoding="utf-8") as f:
    data = json.load(f)
```

The official `co3d` dataclass structure (from `co3d/dataset/data_types.py`) is:

```
FrameAnnotation
├── sequence_name : str
├── frame_number  : int
├── frame_timestamp: float
├── image         : ImageAnnotation(path: str, size: Tuple[int,int])
├── depth         : Optional[DepthAnnotation(path, scale_adjustment, mask_path)]
├── mask          : Optional[MaskAnnotation(path, mass)]
├── viewpoint     : Optional[ViewpointAnnotation(R, T, focal_length, principal_point,
│                                               intrinsics_format)]
└── meta          : Optional[dict]

SequenceAnnotation
├── sequence_name : str
├── category      : str
├── video         : Optional[VideoAnnotation(path, length)]
├── point_cloud   : Optional[PointCloudAnnotation(path, quality_score, n_points)]
└── viewpoint_quality_score : Optional[float]
```

This notebook reads the raw JSON and indexes it defensively with `.get()`, so it keeps working
even if a field is missing in a particular category.

In [10]:
# ---------------------------------------------------------------------------------------------
# Annotation loading (raw gzipped JSON — no hard dependency on the `co3d` package)
# ---------------------------------------------------------------------------------------------

def load_jgz(path: Path) -> List[dict]:
    """Load a gzipped-JSON annotation file into a list of dicts."""
    if not Path(path).exists():
        return []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return json.load(f)


def load_category_annotations(root: Path, category: str) -> Dict[str, Any]:
    """Load and index frame/sequence annotations for one category.

    Frames are indexed by (sequence_name, frame_number) — the *authoritative* identity of a frame.
    Frames are sorted by frame_number inside each sequence, which is the dataset's temporal order.
    """
    cat_dir = root / category
    frames = load_jgz(cat_dir / "frame_annotations.jgz")
    sequences = load_jgz(cat_dir / "sequence_annotations.jgz")

    frames_by_seq: Dict[str, List[dict]] = {}
    for fr in frames:
        frames_by_seq.setdefault(fr.get("sequence_name", ""), []).append(fr)
    for seq_name in frames_by_seq:
        frames_by_seq[seq_name].sort(key=lambda f: f.get("frame_number", 0))

    seq_by_name = {s.get("sequence_name", ""): s for s in sequences}

    return {
        "category": category,
        "frames": frames,
        "sequences": sequences,
        "frames_by_sequence": frames_by_seq,
        "sequence_by_name": seq_by_name,
    }


# Load what is available
ANN = {}
for cat in CONFIG["CATEGORIES"]:
    if (CONFIG["CO3D_ROOT"] / cat).is_dir():
        ANN[cat] = load_category_annotations(CONFIG["CO3D_ROOT"], cat)
        print(f"Loaded annotations for '{cat}': "
              f"{len(ANN[cat]['frames']):,} frames, "
              f"{len(ANN[cat]['sequences']):,} sequences")

if not ANN:
    print("No category annotations available (dataset not downloaded). "
          "EDA cells below will report empty results.")

No category annotations available (dataset not downloaded). EDA cells below will report empty results.


In [11]:
# ---------------------------------------------------------------------------------------------
# EDA 1: category / sequence counts, frame counts, and annotation availability
# ---------------------------------------------------------------------------------------------

def summarise_availability(frames_by_seq: Dict[str, List[dict]], seq_meta: Dict[str, dict],
                           root: Path, category: str) -> pd.DataFrame:
    """Per-sequence record of what annotations are actually present.

    This is a *verification* step: the brief explicitly warns that not every sequence
    contains every annotation, so we check the filesystem per sequence rather than assume.
    """
    rows = []
    for seq_name, frames in frames_by_seq.items():
        seq_dir = root / category / seq_name
        meta = seq_meta.get(seq_name, {})

        n_frames_annotated = len(frames)
        n_images_on_disk = len(list((seq_dir / "images").glob("*"))) if (seq_dir / "images").is_dir() else 0
        n_masks_on_disk = len(list((seq_dir / "masks").glob("*.png"))) if (seq_dir / "masks").is_dir() else 0
        n_depths_on_disk = len(list((seq_dir / "depths").glob("*"))) if (seq_dir / "depths").is_dir() else 0
        n_depth_masks = len(list((seq_dir / "depth_masks").glob("*"))) if (seq_dir / "depth_masks").is_dir() else 0

        # Camera annotations: present on the frame, and complete for all frames?
        vp = [f.get("viewpoint") for f in frames]
        n_with_viewpoint = sum(v is not None for v in vp)
        n_with_intrinsics = sum(
            (v or {}).get("focal_length") is not None and (v or {}).get("principal_point") is not None
            for v in vp
        )

        pc = meta.get("point_cloud") or {}
        rows.append({
            "category": category,
            "sequence_name": seq_name,
            "n_frames_annotated": n_frames_annotated,
            "n_images_on_disk": n_images_on_disk,
            "n_masks_on_disk": n_masks_on_disk,
            "n_depths_on_disk": n_depths_on_disk,
            "n_depth_masks_on_disk": n_depth_masks,
            "n_frames_with_viewpoint": n_with_viewpoint,
            "n_frames_with_intrinsics": n_with_intrinsics,
            "has_pointcloud_file": (seq_dir / "pointcloud.ply").exists(),
            "pointcloud_n_points": pc.get("n_points"),
            "pointcloud_quality": pc.get("quality_score"),
            "viewpoint_quality_score": meta.get("viewpoint_quality_score"),
            "has_video_meta": meta.get("video") is not None,
        })
    return pd.DataFrame(rows)


seq_reports = []
for cat, ann in ANN.items():
    seq_reports.append(
        summarise_availability(ann["frames_by_sequence"], ann["sequence_by_name"],
                               CONFIG["CO3D_ROOT"], cat)
    )

if seq_reports:
    SEQ_REPORT = pd.concat(seq_reports, ignore_index=True)
else:
    SEQ_REPORT = pd.DataFrame(columns=["category", "sequence_name"])

print("=" * 72)
print("EDA 1 — CATEGORIES, SEQUENCES, ANNOTATION AVAILABILITY")
print("=" * 72)
if SEQ_REPORT.empty:
    print("No data available.")
else:
    for cat, grp in SEQ_REPORT.groupby("category"):
        print(f"\n### Category: {cat}")
        print(f"  sequences                       : {len(grp)}")
        print(f"  total annotated frames          : {grp['n_frames_annotated'].sum():,}")
        print(f"  frames/seq  min/median/max      : {grp['n_frames_annotated'].min()} / "
              f"{grp['n_frames_annotated'].median():.0f} / {grp['n_frames_annotated'].max()}")
        print(f"  sequences with masks on disk    : {int((grp['n_masks_on_disk'] > 0).sum())} / {len(grp)}")
        print(f"  sequences with depths on disk   : {int((grp['n_depths_on_disk'] > 0).sum())} / {len(grp)}")
        print(f"  sequences with pointcloud.ply   : {int(grp['has_pointcloud_file'].sum())} / {len(grp)}")
        print(f"  sequences w/ full camera annots : "
              f"{int((grp['n_frames_with_intrinsics'] == grp['n_frames_annotated']).sum())} / {len(grp)}")

    print("\n" + "=" * 72)
    print("EDA 1b — OVERALL AVAILABILITY RATES")
    print("=" * 72)
    n = len(SEQ_REPORT)
    print(f"  masks present     : {int((SEQ_REPORT['n_masks_on_disk'] > 0).sum())}/{n} "
          f"({100*(SEQ_REPORT['n_masks_on_disk'] > 0).mean():.1f}%)")
    print(f"  depths present    : {int((SEQ_REPORT['n_depths_on_disk'] > 0).sum())}/{n} "
          f"({100*(SEQ_REPORT['n_depths_on_disk'] > 0).mean():.1f}%)")
    print(f"  pointcloud present: {int(SEQ_REPORT['has_pointcloud_file'].sum())}/{n} "
          f"({100*SEQ_REPORT['has_pointcloud_file'].mean():.1f}%)")
    print(f"  camera annots full: "
          f"{int((SEQ_REPORT['n_frames_with_intrinsics'] == SEQ_REPORT['n_frames_annotated']).sum())}/{n}")
    print(f"  viewpoint_quality_score available: "
          f"{int(SEQ_REPORT['viewpoint_quality_score'].notna().sum())}/{n}")

EDA 1 — CATEGORIES, SEQUENCES, ANNOTATION AVAILABILITY
No data available.


In [12]:
# ---------------------------------------------------------------------------------------------
# EDA 2: sequence-length distribution + data-quality audit (missing / corrupted files)
# ---------------------------------------------------------------------------------------------

def audit_sequence_files(root: Path, category: str, seq_name: str,
                         frames: List[dict]) -> Dict[str, Any]:
    """Check every annotated frame of one sequence for existence and readability.

    Corruption is detected by actually opening the image with PIL, not by trusting the
    file extension.
    """
    from PIL import Image
    seq_dir = root / category / seq_name
    issues = {"missing": [], "corrupt": [], "size_mismatch": []}
    checked = 0
    sizes = set()
    formats = set()

    for fr in frames:
        img_meta = fr.get("image") or {}
        rel = img_meta.get("path")
        if not rel:
            issues["missing"].append("<no path>")
            continue
        # CO3D image paths in annotations are relative to the category directory.
        p = seq_dir / rel
        if not p.exists():
            p2 = CONFIG["CO3D_ROOT"] / rel
            p = p2 if p2.exists() else p
        if not p.exists():
            issues["missing"].append(str(rel))
            continue
        try:
            with Image.open(p) as im:
                im.verify()                 # cheap structural check
            with Image.open(p) as im:
                w, h = im.size
                fmt = im.format
            sizes.add((w, h))
            formats.add(fmt)
            ann_size = img_meta.get("size")
            if ann_size and tuple(ann_size) != (w, h):
                issues["size_mismatch"].append((str(rel), tuple(ann_size), (w, h)))
            checked += 1
        except Exception as exc:
            issues["corrupt"].append((str(rel), repr(exc)[:80]))

    return {
        "n_checked": checked,
        "n_missing": len(issues["missing"]),
        "n_corrupt": len(issues["corrupt"]),
        "n_size_mismatch": len(issues["size_mismatch"]),
        "image_sizes": sizes,
        "image_formats": formats,
        "issues": issues,
    }


# Audit a representative sample of sequences (full audit is slow on large subsets)
AUDIT_SAMPLE_PER_CATEGORY = 3
audit_rows = []
audit_details = {}

for cat, ann in ANN.items():
    seq_names = sorted(ann["frames_by_sequence"].keys())
    # Deterministic sample
    step = max(1, len(seq_names) // AUDIT_SAMPLE_PER_CATEGORY)
    sample = seq_names[::step][:AUDIT_SAMPLE_PER_CATEGORY]
    for seq_name in sample:
        res = audit_sequence_files(CONFIG["CO3D_ROOT"], cat, seq_name,
                                   ann["frames_by_sequence"][seq_name])
        audit_details[(cat, seq_name)] = res
        audit_rows.append({
            "category": cat,
            "sequence_name": seq_name,
            "n_frames": len(ann["frames_by_sequence"][seq_name]),
            "n_checked": res["n_checked"],
            "n_missing": res["n_missing"],
            "n_corrupt": res["n_corrupt"],
            "n_size_mismatch": res["n_size_mismatch"],
            "sizes": "; ".join(f"{w}x{h}" for w, h in sorted(res["image_sizes"])),
            "formats": "; ".join(sorted(str(f) for f in res["image_formats"])),
        })

AUDIT = pd.DataFrame(audit_rows)

print("=" * 72)
print("EDA 2 — FILE-LEVEL DATA QUALITY AUDIT (sampled sequences)")
print("=" * 72)
if AUDIT.empty:
    print("No data available.")
else:
    print(AUDIT.to_string(index=False))
    print()
    print(f"Total missing images  : {AUDIT['n_missing'].sum()}")
    print(f"Total corrupt images  : {AUDIT['n_corrupt'].sum()}")
    print(f"Size mismatches       : {AUDIT['n_size_mismatch'].sum()}")
    all_sizes = set()
    all_fmts = set()
    for r in audit_details.values():
        all_sizes |= r["image_sizes"]
        all_fmts |= r["image_formats"]
    print(f"Observed image sizes  : {sorted(all_sizes)}")
    print(f"Observed image formats: {sorted(str(f) for f in all_fmts)}")

EDA 2 — FILE-LEVEL DATA QUALITY AUDIT (sampled sequences)
No data available.


In [13]:
# ---------------------------------------------------------------------------------------------
# EDA 3: sequence-length distribution (histogram) + frame-availability table
# ---------------------------------------------------------------------------------------------

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if not SEQ_REPORT.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # (a) sequence length histogram
    ax = axes[0]
    for cat, grp in SEQ_REPORT.groupby("category"):
        ax.hist(grp["n_frames_annotated"], bins=30, alpha=0.6, label=cat, edgecolor="black",
                linewidth=0.4)
    ax.axvline(CONFIG["NUM_FRAMES"], color="crimson", linestyle="--", linewidth=1.6,
               label=f"sample size ({CONFIG['NUM_FRAMES']})")
    ax.set_xlabel("Frames per sequence")
    ax.set_ylabel("Number of sequences")
    ax.set_title("Distribution of sequence lengths")
    ax.legend(fontsize=8)

    # (b) annotation availability per category
    ax = axes[1]
    cats = sorted(SEQ_REPORT["category"].unique())
    features = {
        "masks": lambda g: (g["n_masks_on_disk"] > 0),
        "depths": lambda g: (g["n_depths_on_disk"] > 0),
        "pointcloud": lambda g: g["has_pointcloud_file"],
        "cameras": lambda g: (g["n_frames_with_intrinsics"] == g["n_frames_annotated"]) & (g["n_frames_annotated"] > 0),
    }
    width = 0.2
    for i, (fname, fn) in enumerate(features.items()):
        vals = [fn(SEQ_REPORT[SEQ_REPORT["category"] == c]).mean() * 100 for c in cats]
        ax.bar(np.arange(len(cats)) + i * width, vals, width, label=fname)
    ax.set_xticks(np.arange(len(cats)) + 1.5 * width)
    ax.set_xticklabels(cats)
    ax.set_ylabel("% of sequences with annotation")
    ax.set_title("Annotation availability")
    ax.set_ylim(0, 105)
    ax.legend(fontsize=8, ncol=2)

    fig.tight_layout()
    fig.savefig(DIRS["figures"] / "eda_distribution.png", dpi=120)
    plt.show()
    print(f"Saved: {DIRS['figures'] / 'eda_distribution.png'}")

    # Numeric summary
    print("\nSequence-length summary (per category):")
    print(SEQ_REPORT.groupby("category")["n_frames_annotated"]
          .describe()[["count", "min", "25%", "50%", "75%", "max"]].to_string())
else:
    print("No data available for distribution plot.")

No data available for distribution plot.


In [14]:
# ---------------------------------------------------------------------------------------------
# EDA 4: example frames in chronological order
# ---------------------------------------------------------------------------------------------

def resolve_image_path(root: Path, category: str, seq_name: str, rel_path: str) -> Optional[Path]:
    """Annotation image paths are relative to the category dir, but layouts vary.
    Try the plausible locations and return the first that exists."""
    cands = [
        root / category / seq_name / rel_path,
        root / category / rel_path,
        root / rel_path,
        root / category / seq_name / "images" / Path(rel_path).name,
    ]
    for c in cands:
        if c.exists():
            return c
    return None


def plot_example_frames(root: Path, category: str, seq_name: str,
                        frames: List[dict], n_show: int = 8) -> Optional[Path]:
    """Plot n_show frames of a sequence in chronological (frame_number) order."""
    from PIL import Image
    if not frames:
        return None

    # Even temporal spread across the whole sequence, preserving order.
    idxs = np.linspace(0, len(frames) - 1, min(n_show, len(frames))).astype(int)
    idxs = sorted(set(idxs.tolist()))

    fig, axes = plt.subplots(1, len(idxs), figsize=(2.6 * len(idxs), 3.0))
    if len(idxs) == 1:
        axes = [axes]

    for ax, fi in zip(axes, idxs):
        fr = frames[fi]
        rel = (fr.get("image") or {}).get("path", "")
        p = resolve_image_path(root, category, seq_name, rel)
        if p is not None:
            with Image.open(p) as im:
                ax.imshow(im.convert("RGB"))
        ax.set_title(f"frame {fr.get('frame_number', '?')}", fontsize=8)
        ax.axis("off")

    fig.suptitle(f"{category} / {seq_name}  —  {len(frames)} frames, chronological order",
                 fontsize=11)
    fig.tight_layout()
    out = DIRS["figures"] / f"example_{category}_{seq_name}.png"
    fig.savefig(out, dpi=110)
    plt.close(fig)
    return out


saved_examples = []
for cat, ann in ANN.items():
    seq_names = sorted(ann["frames_by_sequence"].keys())
    if not seq_names:
        continue
    # Pick the longest sequence for the most informative example
    seq_name = max(seq_names, key=lambda s: len(ann["frames_by_sequence"][s]))
    out = plot_example_frames(CONFIG["CO3D_ROOT"], cat, seq_name,
                              ann["frames_by_sequence"][seq_name], n_show=8)
    if out:
        saved_examples.append(out)
        print(f"Saved example frames: {out}  ({category}/{seq_name})")

if not saved_examples:
    print("No example frames could be rendered (dataset images not available).")

No example frames could be rendered (dataset images not available).


In [15]:
# ---------------------------------------------------------------------------------------------
# EDA 5: available train / val / test splits
# ---------------------------------------------------------------------------------------------

def load_set_lists(root: Path, category: str) -> Dict[str, dict]:
    """Load every set_lists_*.json for a category.

    Each file has shape:
        {"train": [(sequence_name, frame_number, image_path), ...],
         "val":   [...],
         "test":  [...]}
    """
    out = {}
    sl_dir = root / category / "set_lists"
    if not sl_dir.is_dir():
        return out
    for f in sorted(sl_dir.glob("*.json")):
        try:
            with open(f, "r", encoding="utf-8") as fh:
                out[f.stem] = json.load(fh)
        except Exception as exc:
            print(f"  [!] could not read {f.name}: {exc}")
    return out


for cat in ANN:
    sls = load_set_lists(CONFIG["CO3D_ROOT"], cat)
    print("=" * 72)
    print(f"SPLITS for category '{cat}'")
    print("=" * 72)
    if not sls:
        print("  No set_lists directory found.")
        continue
    for name, content in sls.items():
        counts = {k: len(v) for k, v in content.items() if isinstance(v, list)}
        seqs = {k: len({t[0] for t in v if isinstance(t, (list, tuple)) and len(t) > 0})
                for k, v in content.items() if isinstance(v, list)}
        print(f"  {name:<40} frames={counts}  sequences={seqs}")

print()
print("Interpretation: the official splits are *frame*-level lists for the many-view / few-view")
print("reconstruction protocols. For a *sequence-level* quality-prediction task they are not used")
print("directly for the final split — instead Section 6 splits whole sequences by object identity,")
print("which is strictly more conservative and prevents frame-level leakage.")


Interpretation: the official splits are *frame*-level lists for the many-view / few-view
reconstruction protocols. For a *sequence-level* quality-prediction task they are not used
directly for the final split — instead Section 6 splits whole sequences by object identity,
which is strictly more conservative and prevents frame-level leakage.


## 5. Ground-truth label generation (COLMAP reconstruction experiments)

This is the core scientific contribution of the project. CO3D gives no universal Good/Poor label,
so we **construct** one by measuring what a real Structure-from-Motion reconstruction actually achieves.

### 5.1 The label-generation procedure

For each candidate sequence:

1. **Sample frames** in temporal order (default 16 — the same count the model will see at inference).
2. **Write frames to a COLMAP workspace** (`workspace/images/`), preserving the sampled order.
3. **Run COLMAP**: `feature_extractor` → `exhaustive_matcher` → `mapper`.
4. **Harvest metrics** from the sparse model:
   - `registration_ratio` = registered images / sampled images
   - `mean_reprojection_error` = mean per-point reprojection error in pixels (from `points3D.txt`)
   - `n_registered_cameras`
   - `n_points_3d`, `mean_track_length`, `point_density` (points per registered image)
   - `coverage` — fraction of the reference surface (CO3D `pointcloud.ply` / depth maps) explained
     by the reconstructed points
   - `reconstruction_status` — `success` / `partial` / `failed`
5. **Combine into a 0–100 quality score** (Section 5.4).
6. **Derive the Good/Poor label** from the score using a validation-calibrated threshold.

### 5.2 Why COLMAP needs no ground-truth poses

A key subtlety: COLMAP must be run **without** CO3D's ground-truth camera poses, otherwise it is not
reconstructing anything — it is just being handed the answer. The pipeline therefore deliberately
does **not** write a `cameras.txt` with GT intrinsics. COLMAP estimates intrinsics itself, which is
exactly the situation the deployed system will face with an arbitrary new video.

### 5.3 Handling failures — the rule

The brief asks that failures are treated **separately** rather than assigned an arbitrary score. The
rule implemented here:

| Condition | Status | Score |
|---|---|---|
| `n_registered < COLMAP_MIN_NUM_REG` (default 3) | `failed` | **0** — a reconstruction that cannot produce a usable model |
| Registered but weak geometry | `partial` | Composite score, capped at 40 |
| Registered with good geometry | `success` | Composite score (0–100) |

A sequence that COLMAP cannot solve is *genuinely* a Poor candidate for reconstruction, so scoring it
0 is not "arbitrary" — it is a definitional statement. It is recorded with `status='failed'` so that
failures can be filtered or modelled separately later.

### 5.4 The 0–100 quality score

Four normalised components, each in [0, 1], combined with documented weights:

| Component | Normalisation | Weight | Rationale |
|---|---|---|---|
| `registration_ratio` | already in [0, 1] | **0.35** | The single most direct proxy for "will reconstruction succeed". |
| `reproj_error` | `exp(-e / e0)`, `e0 = 2 px` | **0.25** | Geometric accuracy; lower error ⇒ better. Exponential keeps the penalty smooth and bounded. |
| `point_density` | `min(1, pts / (10 × n_reg))` | **0.20** | Distinguishes a dense, well-triangulated model from a degenerate one. |
| `coverage` | fraction of reference surface covered | **0.20** | Rewards genuine *coverage* of the object, not just a few scattered points. |

```
score = 100 × (0.35·reg + 0.25·exp_err + 0.20·density + 0.20·coverage)
```

Weights are exposed in `CONFIG["SCORE_WEIGHTS"]` and can be justified or re-tuned on the validation set.

In [16]:
# ---------------------------------------------------------------------------------------------
# COLMAP driver — runs the real SfM pipeline and harvests metrics.
# ---------------------------------------------------------------------------------------------

def colmap_available() -> bool:
    return shutil.which("colmap") is not None


def run_cmd(cmd: List[str], log_path: Path, timeout: int = 3600) -> Dict[str, Any]:
    """Run a shell command, tee output to a log file, and never raise."""
    log_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, timeout=timeout)
        with open(log_path, "w", encoding="utf-8") as f:
            f.write(proc.stdout or "")
        return {"ok": proc.returncode == 0, "returncode": proc.returncode,
                "output": proc.stdout or "", "log": str(log_path)}
    except subprocess.TimeoutExpired:
        with open(log_path, "w", encoding="utf-8") as f:
            f.write("TIMEOUT")
        return {"ok": False, "returncode": -9, "output": "TIMEOUT", "log": str(log_path)}
    except Exception as exc:
        with open(log_path, "w", encoding="utf-8") as f:
            f.write(repr(exc))
        return {"ok": False, "returncode": -1, "output": repr(exc), "log": str(log_path)}


def parse_colmap_model(model_dir: Path) -> Dict[str, Any]:
    """Parse a COLMAP sparse model (text format) and compute raw metrics.

    Text format is used because it is human-readable and stable across COLMAP versions:
      images.txt  : 2 lines per image; line 1 = pose, line 2 = POINTS2D triples
      points3D.txt: one line per point: id X Y Z R G B ERROR track...
    """
    metrics: Dict[str, Any] = {
        "n_registered_cameras": 0,
        "n_points_3d": 0,
        "mean_reprojection_error": None,
        "median_reprojection_error": None,
        "mean_track_length": 0.0,
        "registered_image_names": [],
        "n_observations": 0,
    }

    images_txt = model_dir / "images.txt"
    points_txt = model_dir / "points3D.txt"

    # --- images ---
    if images_txt.exists():
        with open(images_txt, "r", encoding="utf-8") as f:
            lines = [ln.rstrip("\n") for ln in f if not ln.startswith("#")]
        # Non-comment lines: pose lines are at even indices, POINTS2D at odd indices.
        for i in range(0, len(lines) - 1, 2):
            parts = lines[i].split()
            if len(parts) >= 10:
                metrics["registered_image_names"].append(parts[9])
            # count valid 2D observations that triangulated into a 3D point
            pts2d = lines[i + 1].split()
            metrics["n_observations"] += sum(1 for j in range(0, len(pts2d), 3)
                                             if j + 2 < len(pts2d) and pts2d[j + 2] != "-1")
        metrics["n_registered_cameras"] = len(metrics["registered_image_names"])

    # --- points3D ---
    if points_txt.exists():
        errors, tracks = [], []
        with open(points_txt, "r", encoding="utf-8") as f:
            for ln in f:
                if ln.startswith("#"):
                    continue
                p = ln.split()
                if len(p) < 8:
                    continue
                try:
                    errors.append(float(p[7]))
                    tracks.append(max(0, (len(p) - 8) // 2))
                except ValueError:
                    continue
        metrics["n_points_3d"] = len(errors)
        if errors:
            metrics["mean_reprojection_error"] = float(np.mean(errors))
            metrics["median_reprojection_error"] = float(np.median(errors))
        if tracks:
            metrics["mean_track_length"] = float(np.mean(tracks))

    return metrics


def find_model_dir(workspace: Path) -> Optional[Path]:
    """COLMAP writes sparse models to sparse/0, sparse/1, ... — take the largest one."""
    sparse = workspace / "sparse"
    if not sparse.is_dir():
        return None
    cands = [d for d in sparse.iterdir() if d.is_dir() and (d / "images.bin").exists() or (d / "images.txt").exists()]
    if not cands:
        return None
    # Prefer text format for parsing
    text_models = [d for d in cands if (d / "images.txt").exists()]
    pool = text_models if text_models else cands
    return max(pool, key=lambda d: parse_colmap_model(d)["n_registered_cameras"])


def reconstruct_sequence(frames: List[dict], category: str, seq_name: str,
                         root: Path, n_images: int = 16,
                         force: bool = False) -> Dict[str, Any]:
    """Run a full COLMAP SfM reconstruction for one sequence and return its metrics.

    Returns a metrics dict. On any failure the dict contains ``reconstruction_status``
    of 'failed' and ``error`` describing what went wrong — the caller records the exclusion
    rather than crashing.
    """
    result: Dict[str, Any] = {
        "category": category,
        "sequence_name": seq_name,
        "n_sampled_frames": 0,
        "n_registered_cameras": 0,
        "registration_ratio": 0.0,
        "mean_reprojection_error": None,
        "n_points_3d": 0,
        "mean_track_length": 0.0,
        "point_density": 0.0,
        "coverage": None,
        "reconstruction_status": "failed",
        "error": None,
        "sampled_frame_paths": [],
    }

    # --- pick frames in temporal order ---
    ordered = sorted(frames, key=lambda f: f.get("frame_number", 0))
    if len(ordered) > n_images:
        idxs = np.linspace(0, len(ordered) - 1, n_images).astype(int)
        ordered = [ordered[i] for i in sorted(set(idxs.tolist()))]
    result["n_sampled_frames"] = len(ordered)
    if len(ordered) < 3:
        result["error"] = "fewer than 3 usable frames"
        return result

    # --- prepare workspace ---
    ws = DIRS["colmap"] / category / seq_name
    img_dir = ws / "images"
    if force and ws.exists():
        shutil.rmtree(ws, ignore_errors=True)
    img_dir.mkdir(parents=True, exist_ok=True)

    # --- copy frames in with a stable, ordered name ---
    ok_paths = []
    for i, fr in enumerate(ordered):
        rel = (fr.get("image") or {}).get("path", "")
        src = resolve_image_path(root, category, seq_name, rel)
        if src is None:
            continue
        dst = img_dir / f"frame{i:05d}.jpg"
        try:
            shutil.copyfile(src, dst)
            ok_paths.append(str(dst))
        except Exception:
            continue

    result["sampled_frame_paths"] = ok_paths
    if len(ok_paths) < 3:
        result["error"] = "fewer than 3 images could be copied into the COLMAP workspace"
        return result

    if not colmap_available():
        result["error"] = "colmap executable not found on PATH"
        result["reconstruction_status"] = "unavailable"
        return result

    db = ws / "database.db"
    if force and db.exists():
        db.unlink()

    # 1) feature extraction
    r = run_cmd([
        "colmap", "feature_extractor",
        "--database_path", str(db),
        "--image_path", str(img_dir),
        "--ImageReader.single_camera", "1",
        "--SiftExtraction.use_gpu", "1" if _gpu_available() else "0",
    ], ws / "log_feature.txt")
    if not r["ok"]:
        result["error"] = f"feature_extractor failed (rc={r['returncode']})"
        return result

    # 2) exhaustive matching
    r = run_cmd([
        "colmap", "exhaustive_matcher",
        "--database_path", str(db),
        "--SiftMatching.use_gpu", "1" if _gpu_available() else "0",
    ], ws / "log_match.txt")
    if not r["ok"]:
        result["error"] = f"exhaustive_matcher failed (rc={r['returncode']})"
        return result

    # 3) sparse reconstruction
    r = run_cmd([
        "colmap", "mapper",
        "--database_path", str(db),
        "--image_path", str(img_dir),
        "--output_path", str(ws / "sparse"),
        "--Mapper.ba_global_function_tolerance", "0.0001",
    ], ws / "log_mapper.txt")
    if not r["ok"]:
        result["error"] = f"mapper failed (rc={r['returncode']})"
        return result

    # 4) harvest metrics
    model_dir = find_model_dir(ws)
    if model_dir is None:
        result["error"] = "no sparse model produced"
        return result

    m = parse_colmap_model(model_dir)
    result.update({
        "n_registered_cameras": m["n_registered_cameras"],
        "registration_ratio": m["n_registered_cameras"] / max(1, len(ok_paths)),
        "mean_reprojection_error": m["mean_reprojection_error"],
        "median_reprojection_error": m["median_reprojection_error"],
        "n_points_3d": m["n_points_3d"],
        "mean_track_length": m["mean_track_length"],
        "n_observations": m["n_observations"],
    })
    result["point_density"] = m["n_points_3d"] / max(1, m["n_registered_cameras"])

    n_reg = m["n_registered_cameras"]
    if n_reg < CONFIG["COLMAP_MIN_NUM_REG"]:
        result["reconstruction_status"] = "failed"
    elif result["registration_ratio"] < 0.6:
        result["reconstruction_status"] = "partial"
    else:
        result["reconstruction_status"] = "success"
    return result


def _gpu_available() -> bool:
    try:
        import torch
        return bool(torch.cuda.is_available())
    except Exception:
        return False


print("COLMAP available:", colmap_available())
print("COLMAP workspace root:", DIRS["colmap"])

COLMAP available: True
COLMAP workspace root: /content/work/colmap


In [17]:
# ---------------------------------------------------------------------------------------------
# Coverage metric — how much of the reference object surface did the reconstruction explain?
# ---------------------------------------------------------------------------------------------

def read_ply_points(path: Path, max_points: int = 200000) -> Optional[np.ndarray]:
    """Read an ASCII or binary PLY into an (N, 3) array. Returns None on failure."""
    if not Path(path).exists():
        return None
    try:
        from plyfile import PlyData
        ply = PlyData.read(str(path))
        v = ply["vertex"]
        xyz = np.stack([v["x"], v["y"], v["z"]], axis=1).astype(np.float64)
        return xyz[:max_points]
    except Exception:
        pass
    # Minimal ASCII PLY fallback (no third-party dependency)
    try:
        pts = []
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            in_vertex = False
            n_vertex = 0
            header_done = False
            for ln in f:
                ln = ln.strip()
                if not header_done:
                    if ln.startswith("element vertex"):
                        in_vertex = True
                        n_vertex = int(ln.split()[-1])
                    elif ln.startswith("end_header"):
                        header_done = True
                    continue
                if in_vertex:
                    p = ln.split()
                    if len(p) >= 3:
                        pts.append([float(p[0]), float(p[1]), float(p[2])])
                    if len(pts) >= max_points:
                        break
        if pts:
            return np.asarray(pts, dtype=np.float64)
    except Exception:
        pass
    return None


def compute_coverage(recon_points: Optional[np.ndarray],
                     ref_points: Optional[np.ndarray],
                     voxel: float = 0.01) -> Optional[float]:
    """Fraction of reference-surface points that are explained by the reconstruction.

    Both point sets are voxelised; a reference voxel counts as covered when the
    reconstruction has a point within `voxel` distance of it. Returns None when
    a reference cloud is unavailable (the brief asks for coverage "where a reliable
    reference is available" — it is not always).
    """
    if recon_points is None or ref_points is None:
        return None
    if len(recon_points) == 0 or len(ref_points) == 0:
        return None
    try:
        from scipy.spatial import cKDTree
        tree = cKDTree(recon_points)
        # For each reference point, distance to the nearest reconstructed point
        d, _ = tree.query(ref_points, k=1)
        return float(np.mean(d <= voxel))
    except Exception:
        # Fallback: voxel-hash overlap (no scipy)
        rv = set(map(tuple, np.floor(ref_points / voxel).astype(np.int64)))
        cv = set(map(tuple, np.floor(recon_points / voxel).astype(np.int64)))
        if not rv:
            return None
        return float(len(rv & cv) / len(rv))


print("Coverage helper ready.")
print("Install optional extras for full PLY support:  pip install plyfile scipy")

Coverage helper ready.
Install optional extras for full PLY support:  pip install plyfile scipy


In [18]:
# ---------------------------------------------------------------------------------------------
# Composite 0-100 quality score + Good/Poor label
# ---------------------------------------------------------------------------------------------

def normalise_reprojection_error(err: Optional[float], e0: float = 2.0) -> float:
    """Lower error -> higher quality. Bounded, smooth, monotone decreasing."""
    if err is None or not np.isfinite(err) or err <= 0:
        return 0.0
    return float(np.exp(-err / e0))


def normalise_point_density(density: float, n_reg: int, target_per_image: float = 10.0) -> float:
    """Normalise points-per-registered-image into [0, 1]."""
    if n_reg <= 0:
        return 0.0
    return float(min(1.0, density / target_per_image))


def compute_quality_score(metrics: Dict[str, Any],
                           weights: Optional[Dict[str, float]] = None) -> Dict[str, Any]:
    """Combine raw COLMAP metrics into a documented 0-100 score and a Good/Poor label.

    Failure rule (see Section 5.3):
      * status 'failed'      -> score 0, label Poor
      * status 'partial'     -> composite score, capped at 40, label Poor
      * status 'success'     -> composite score (0-100), label by threshold
    """
    w = dict(weights or CONFIG["SCORE_WEIGHTS"])
    assert abs(sum(w.values()) - 1.0) < 1e-6, f"Weights must sum to 1.0, got {sum(w.values())}"

    status = metrics.get("reconstruction_status", "failed")
    n_reg = int(metrics.get("n_registered_cameras", 0) or 0)

    if status in ("failed", "unavailable"):
        return {"quality_score": 0.0, "label": "Poor", "status": status,
                "components": {"registration_ratio": 0.0, "reproj_error": 0.0,
                               "point_density": 0.0, "coverage": 0.0}}

    reg = float(np.clip(metrics.get("registration_ratio", 0.0), 0.0, 1.0))
    err_c = normalise_reprojection_error(metrics.get("mean_reprojection_error"))
    dens_c = normalise_point_density(metrics.get("point_density", 0.0), n_reg)
    cov = metrics.get("coverage")
    cov_c = float(np.clip(cov, 0.0, 1.0)) if cov is not None else 0.0
    # A missing coverage reference should not silently penalise the sequence.
    coverage_used = cov is not None

    comps = {"registration_ratio": reg, "reproj_error": err_c,
             "point_density": dens_c, "coverage": cov_c}

    if coverage_used:
        score = 100.0 * sum(w[k] * comps[k] for k in w)
    else:
        # Renormalise the available components to preserve the weight ratio.
        avail = {k: v for k, v in w.items() if k != "coverage"}
        s = sum(avail.values())
        score = 100.0 * sum((avail[k] / s) * comps[k] for k in avail)

    if status == "partial":
        score = min(score, 40.0)

    score = float(np.clip(score, 0.0, 100.0))
    label = "Good" if score >= CONFIG["GOOD_SCORE_THRESHOLD"] else "Poor"
    return {"quality_score": score, "label": label, "status": status, "components": comps}


print("Score function ready. Weights:", CONFIG["SCORE_WEIGHTS"])
print("Threshold (provisional):", CONFIG["GOOD_SCORE_THRESHOLD"])

Score function ready. Weights: {'registration_ratio': 0.35, 'reproj_error': 0.25, 'n_points_density': 0.2, 'coverage': 0.2}
Threshold (provisional): 50.0


In [19]:
# ---------------------------------------------------------------------------------------------
# Run label generation over selected sequences, with resumable caching.
# ---------------------------------------------------------------------------------------------

# How many sequences per category to attempt. Start small — COLMAP is slow.
SEQUENCES_PER_CATEGORY = int(os.environ.get("SEQUENCES_PER_CATEGORY", "8"))
FORCE_RERUN = False

label_rows: List[Dict[str, Any]] = []

if not ANN:
    print("No annotations loaded — skipping label generation.")
else:
    for cat, ann in ANN.items():
        seq_names = sorted(ann["frames_by_sequence"].keys())
        if not seq_names:
            continue
        # Prefer sequences that actually have images on disk.
        usable = [s for s in seq_names
                  if (CONFIG["CO3D_ROOT"] / cat / s / "images").is_dir()]
        if not usable:
            usable = seq_names
        # Deterministic selection spread over the category
        if len(usable) > SEQUENCES_PER_CATEGORY:
            idxs = np.linspace(0, len(usable) - 1, SEQUENCES_PER_CATEGORY).astype(int)
            usable = [usable[i] for i in sorted(set(idxs.tolist()))]

        print("=" * 72)
        print(f"LABEL GENERATION — category '{cat}' ({len(usable)} sequences)")
        print("=" * 72)

        for seq_name in usable:
            cache = DIRS["labels"] / f"{cat}__{seq_name}.json"
            if cache.exists() and not FORCE_RERUN:
                with open(cache, "r", encoding="utf-8") as f:
                    row = json.load(f)
                row["from_cache"] = True
                label_rows.append(row)
                print(f"  [cache] {seq_name:<28} score={row.get('quality_score', 0):6.2f} "
                      f"label={row.get('label')} status={row.get('reconstruction_status')}")
                continue

            frames = ann["frames_by_sequence"][seq_name]
            m = reconstruct_sequence(frames, cat, seq_name, CONFIG["CO3D_ROOT"],
                                     n_images=CONFIG["COLMAP_MAX_IMAGES"], force=FORCE_RERUN)

            # Coverage vs the CO3D reference point cloud, when available.
            if m.get("reconstruction_status") in ("success", "partial"):
                model_dir = find_model_dir(DIRS["colmap"] / cat / seq_name)
                recon_pts = None
                if model_dir is not None:
                    p3d = model_dir / "points3D.txt"
                    if p3d.exists():
                        pts = []
                        with open(p3d, "r", encoding="utf-8") as f:
                            for ln in f:
                                if ln.startswith("#"):
                                    continue
                                p = ln.split()
                                if len(p) >= 5:
                                    try:
                                        pts.append([float(p[1]), float(p[2]), float(p[3])])
                                    except ValueError:
                                        pass
                        recon_pts = np.asarray(pts) if pts else None
                ref_pts = read_ply_points(CONFIG["CO3D_ROOT"] / cat / seq_name / "pointcloud.ply")
                cov = compute_coverage(recon_pts, ref_pts)
                m["coverage"] = cov

            scored = compute_quality_score(m)
            row = {**m, **{k: v for k, v in scored.items() if k != "components"},
                   "score_components": scored["components"],
                   "from_cache": False}
            with open(cache, "w", encoding="utf-8") as f:
                json.dump(row, f, indent=2)
            label_rows.append(row)
            print(f"  [run]   {seq_name:<28} score={row['quality_score']:6.2f} "
                  f"label={row['label']:<4} status={row['reconstruction_status']:<12} "
                  f"reg={row['registration_ratio']:.2f} "
                  f"pts={row['n_points_3d']}")

# ---- Persist the metadata CSV required by the brief ----
LABELS_CSV = DIRS["labels"] / "sequence_labels.csv"
if label_rows:
    label_df = pd.DataFrame(label_rows)
    # Expand the component dict for readability
    for c in ["registration_ratio", "reproj_error", "point_density", "coverage"]:
        if "score_components" in label_df.columns:
            label_df[f"comp_{c}"] = label_df["score_components"].apply(
                lambda d: (d or {}).get(c))
    keep = [c for c in [
        "category", "sequence_name", "n_sampled_frames", "n_registered_cameras",
        "registration_ratio", "mean_reprojection_error", "n_points_3d",
        "mean_track_length", "point_density", "coverage",
        "quality_score", "label", "reconstruction_status", "error",
        "sampled_frame_paths",
    ] if c in label_df.columns]
    label_df[keep].to_csv(LABELS_CSV, index=False)
    print(f"\nSaved label metadata CSV -> {LABELS_CSV}")
    print(label_df[keep].to_string(index=False))
else:
    print("\nNo labels produced (no data or COLMAP unavailable).")

No annotations loaded — skipping label generation.

No labels produced (no data or COLMAP unavailable).


### 5.5 Label-generation limitations (stated up front)

These are real and they bound what the model can learn:

1. **Label ≠ 3D quality.** The score measures *how well COLMAP did on these frames*, not absolute
   3D accuracy. A sequence that reconstructs beautifully but is an odd object still scores high.
2. **COLMAP configuration dependency.** Scores are tied to the COLMAP version, SIFT settings, and
   GPU/CPU nondeterminism. Re-running with different settings shifts scores. Mitigated by fixing
   the pipeline and caching results.
3. **Coverage is often unavailable.** Reference point clouds are missing for some sequences; the
   weight renormalisation in `compute_quality_score` handles this, but the effective metric mix
   then differs slightly between sequences.
4. **Small sample.** Dozens-to-hundreds of labelled sequences is a small regression problem. The
   continuous score is the primary target; the binary label inherits its noise.
5. **Threshold is provisional.** The 50-point cut is a starting point. Section 6 recalibrates it on
   the validation split only.
6. **Survivorship.** Sequences that fail to even load (missing images) are excluded entirely, which
   biases the label distribution towards "processable" sequences. These exclusions are recorded
   in `labels/exclusions.csv`.

## 6. Data quality checks and sequence-level splitting

### 6.1 Quality checks

Before training, every labelled sequence is validated:
- image files present and readable
- frame count sufficient for `NUM_FRAMES` sampling (shorter ones are padded, not dropped)
- no NaN/inf in the quality score
- quality score within [0, 100]

Excluded sequences are written to an exclusions CSV — the brief requires all exclusions to be recorded.

### 6.2 Splitting without leakage

The split is performed at the **sequence level**, and where the underlying object identity is
recoverable, at the **object level** too. Crucially:

> **Frames from the same sequence never appear in more than one split**, and sequences of the same
> physical object are grouped together. The split is computed *before* any model sees data.

The brief's target proportions are 70 / 15 / 15, adjusted when the labelled set is small.

In [20]:
# ---------------------------------------------------------------------------------------------
# Data quality gate + sequence-level split
# ---------------------------------------------------------------------------------------------

def check_and_filter(labels: List[dict], ann: Dict[str, Any], root: Path,
                     category: str, min_frames: int = 8) -> Tuple[List[dict], List[dict]]:
    """Split labelled rows into (kept, excluded) based on hard data-quality rules."""
    kept, excluded = [], []
    for row in labels:
        seq_name = row.get("sequence_name")
        problems = []

        frames = ann["frames_by_sequence"].get(seq_name, [])
        if len(frames) < min_frames:
            problems.append(f"too few frames ({len(frames)} < {min_frames})")

        img_dir = root / category / seq_name / "images"
        n_imgs = len(list(img_dir.glob("*"))) if img_dir.is_dir() else 0
        if n_imgs == 0:
            problems.append("no images on disk")

        qs = row.get("quality_score")
        if qs is None or not np.isfinite(qs):
            problems.append("non-finite quality score")
        elif not (0.0 <= float(qs) <= 100.0):
            problems.append(f"quality score out of range: {qs}")

        if row.get("reconstruction_status") == "unavailable":
            problems.append("COLMAP unavailable — no real label")

        if problems:
            excluded.append({**row, "exclusion_reason": "; ".join(problems)})
        else:
            kept.append(row)
    return kept, excluded


if label_rows:
    kept_rows, excluded_rows = [], []
    for cat, ann in ANN.items():
        cat_rows = [r for r in label_rows if r.get("category") == cat]
        k, e = check_and_filter(cat_rows, ann, CONFIG["CO3D_ROOT"], cat)
        kept_rows += k
        excluded_rows += e

    EXCLUSIONS = pd.DataFrame(excluded_rows)
    if not EXCLUSIONS.empty:
        EXCLUSIONS.to_csv(DIRS["labels"] / "exclusions.csv", index=False)
        print(f"Excluded {len(EXCLUSIONS)} sequences -> {DIRS['labels'] / 'exclusions.csv'}")
        print(EXCLUSIONS[["category", "sequence_name", "exclusion_reason"]].to_string(index=False))

    LABEL_DF = pd.DataFrame(kept_rows)
else:
    LABEL_DF = pd.DataFrame(columns=["category", "sequence_name", "quality_score", "label"])
    EXCLUSIONS = pd.DataFrame()

print(f"\nUsable labelled sequences: {len(LABEL_DF)}")
if not LABEL_DF.empty:
    print("\nLabel distribution:")
    print(LABEL_DF["label"].value_counts().to_string())
    print(f"\nQuality score stats:\n{LABEL_DF['quality_score'].describe().to_string()}")
    if LABEL_DF["label"].nunique() < 2:
        print("\n[!] WARNING: only one class present. A binary classifier cannot be trained.")
        print("    Options: (a) run COLMAP on more sequences, (b) lower the quality threshold,")
        print("    (c) lower COLMAP_MIN_NUM_REG, (d) train the regression head only.")


Usable labelled sequences: 0


In [21]:
# ---------------------------------------------------------------------------------------------
# Sequence-level split (grouped by object identity when available)
# ---------------------------------------------------------------------------------------------

def object_group_key(row: dict) -> str:
    """Best-effort grouping key for 'the same physical object'.

    Priority:
      1. An explicit object/id field if the annotation carries one.
      2. The sequence's video basename (CO3D videos are typically one per object).
      3. The sequence name itself (always safe — worst case = no grouping beyond sequence).
    """
    seq = row.get("sequence_name", "")
    for key in ("object_id", "object_name", "scene_name", "instance_id"):
        if row.get(key):
            return f"{row.get('category','?')}::{row[key]}"
    return f"{row.get('category','?')}::{seq}"


def sequence_level_split(df: pd.DataFrame, train_frac: float = 0.70,
                          val_frac: float = 0.15, test_frac: float = 0.15,
                          seed: int = SEED) -> Dict[str, List[str]]:
    """Split whole sequences (grouped by object identity) into train/val/test.

    Greedy group-wise allocation: groups are shuffled deterministically and then assigned
    to whichever split is furthest below its target count. This keeps proportions close
    to the requested ones even when group sizes are uneven.
    """
    if df.empty:
        return {"train": [], "val": [], "test": []}

    work = df.copy()
    work["_group"] = work.apply(object_group_key, axis=1)

    groups: Dict[str, List[str]] = {}
    for g, grp in work.groupby("_group"):
        groups[g] = sorted(grp["sequence_name"].tolist())

    rng = np.random.default_rng(seed)
    order = list(groups.keys())
    rng.shuffle(order)
    # Largest groups first improves the chance of hitting target proportions.
    order.sort(key=lambda g: -len(groups[g]))

    targets = {"train": train_frac, "val": val_frac, "test": test_frac}
    total = len(work)
    quota = {k: v * total for k, v in targets.items()}
    filled = {k: 0 for k in targets}

    out: Dict[str, List[str]] = {"train": [], "val": [], "test": []}
    for g in order:
        # assign to the split with the largest remaining deficit
        split = max(targets.keys(), key=lambda k: quota[k] - filled[k])
        out[split].extend(groups[g])
        filled[split] += len(groups[g])

    for k in out:
        out[k] = sorted(out[k])
    return out


SPLITS = sequence_level_split(LABEL_DF) if not LABEL_DF.empty else {"train": [], "val": [], "test": []}

if LABEL_DF.empty:
    print("No labelled data — split is empty.")
else:
    print("=" * 72)
    print("SEQUENCE-LEVEL SPLIT")
    print("=" * 72)
    for split, names in SPLITS.items():
        pct = 100 * len(names) / len(LABEL_DF)
        print(f"  {split:<6} {len(names):>4} sequences  ({pct:5.1f}%)")

    # ---- Leakage verification ----
    print("\n--- Leakage verification ---")
    all_sets = [set(SPLITS[k]) for k in SPLITS]
    overlaps = []
    for i in range(len(all_sets)):
        for j in range(i + 1, len(all_sets)):
            ov = all_sets[i] & all_sets[j]
            if ov:
                overlaps.append(ov)
    print(f"  sequence overlap between splits : {len(overlaps)}  "
          f"{'PASS' if not overlaps else 'FAIL -> ' + str(overlaps)}")

    # object-group overlap
    key_by_seq = {r["sequence_name"]: object_group_key(r) for _, r in LABEL_DF.iterrows()}
    gsets = [{key_by_seq[s] for s in SPLITS[k]} for k in SPLITS]
    g_ov = []
    for i in range(len(gsets)):
        for j in range(i + 1, len(gsets)):
            ov = gsets[i] & gsets[j]
            if ov:
                g_ov.append(ov)
    print(f"  object-group overlap between splits: {len(g_ov)}  "
          f"{'PASS' if not g_ov else 'FAIL -> ' + str(g_ov)}")

    # frame-level check: no frame path may appear in two splits
    path_by_seq = {}
    for _, r in LABEL_DF.iterrows():
        path_by_seq[r["sequence_name"]] = set(r.get("sampled_frame_paths") or [])
    f_ov = []
    for i in range(len(SPLITS)):
        for j in range(i + 1, len(SPLITS)):
            pa = set().union(*[path_by_seq.get(s, set()) for s in SPLITS[SPLITS[i]]] or [set()]) if SPLITS[i] else set()
            pb = set().union(*[path_by_seq.get(s, set()) for s in SPLITS[SPLITS[i + 1]]] or [set()]) if SPLITS[i + 1] else set()
            ov = pa & pb
            if ov:
                f_ov.append(ov)
    print(f"  frame-path overlap between splits : {len(f_ov)}  "
          f"{'PASS' if not f_ov else 'FAIL'}")

    # ---- Threshold calibration on VALIDATION only ----
    print("\n--- Good/Poor threshold calibration (validation split only) ---")
    val_scores = LABEL_DF[LABEL_DF["sequence_name"].isin(SPLITS["val"])].get("quality_score", pd.Series(dtype=float))
    if len(val_scores) >= 4:
        # Choose the threshold maximising validation F1 over a grid of candidate splits.
        from sklearn.metrics import f1_score
        grid = np.arange(10, 91, 5)
        y_true = (val_scores.values >= 50).astype(int)
        best = (-1, None)
        for t in grid:
            y_pred = (val_scores.values >= t).astype(int)
            if y_true.sum() == 0 or y_pred.sum() == 0:
                continue
            f1 = f1_score(y_true, y_pred, zero_division=0)
            if f1 > best[0]:
                best = (f1, float(t))
        if best[1] is not None:
            CONFIG["GOOD_SCORE_THRESHOLD"] = best[1]
            print(f"  Selected threshold = {best[1]:.0f} (validation F1 = {best[0]:.3f})")
            print("  -> labels in the CSV are re-derived from this threshold below.")
    else:
        print(f"  Only {len(val_scores)} validation sequences — too few to calibrate reliably.")
        print(f"  Keeping provisional threshold = {CONFIG['GOOD_SCORE_THRESHOLD']:.0f}")

    # Re-derive labels with the final threshold (train+val+test labels all updated;
    # the threshold itself came from validation only)
    LABEL_DF["label"] = np.where(
        LABEL_DF["quality_score"] >= CONFIG["GOOD_SCORE_THRESHOLD"], "Good", "Poor")
    LABEL_DF["split"] = LABEL_DF["sequence_name"].map(
        {s: k for k, names in SPLITS.items() for s in names})
    LABEL_DF.to_csv(DIRS["labels"] / "sequence_labels_split.csv", index=False)
    print(f"\nSaved split labels -> {DIRS['labels'] / 'sequence_labels_split.csv'}")
    print(LABEL_DF.groupby(["split", "label"]).size().to_string())

No labelled data — split is empty.


## 7. Frame sampling and preprocessing

### 7.1 Sampling rules

- One **sequence** = one training example (not one frame).
- Frames are ordered by `frame_number` from the annotations — **not** by filename, because CO3D
  filenames do not reliably match frame numbers.
- `NUM_FRAMES` frames are sampled with an **even temporal spread** across the sequence, preserving order.
- Frames are resized to **224 × 224** and normalised with the ResNet-18 / ImageNet statistics.

### 7.2 Short sequences — the padding/masking strategy

A sequence with fewer than `NUM_FRAMES` frames is **not** dropped. It is padded by repeating its
**last** frame, and a boolean **validity mask** of shape `[seq_len]` is produced alongside the
tensor. The mask is passed to the LSTM's `pack_padded_sequence`, so padded frames contribute
**nothing** to the temporal state. This keeps short sequences usable without letting padding
pollute the learned representation.

### 7.3 Output tensor shapes

```
frames      : [batch, seq_len, 3, 224, 224]      float32
mask        : [batch, seq_len]                   bool
target      : [batch]                            float32   (quality score 0-100)
```

A batch of 8 sequences × 16 frames gives `[8, 16, 3, 224, 224]`.

In [22]:
# ---------------------------------------------------------------------------------------------
# Preprocessing pipeline
# ---------------------------------------------------------------------------------------------

try:
    import torch
    import torch.nn as nn
    import torchvision
    from torchvision import transforms
    TORCH_OK = True
except Exception as exc:                                    # pragma: no cover
    TORCH_OK = False
    print(f"[!] PyTorch unavailable: {exc}")

# ImageNet statistics — must match the pretrained ResNet-18 exactly.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def build_transform(img_size: int = 224) -> "transforms.Compose":
    """Deterministic training-time preprocessing (no random augmentation here —
    augmentation is a documented extension point, see Section 10.3)."""
    return transforms.Compose([
        transforms.Resize((img_size, img_size), interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def sample_frame_indices(n_available: int, n_want: int) -> List[int]:
    """Even temporal spread over the sequence, in ascending order.

    Uses linspace so the sampled frames cover the full temporal extent rather than
    clustering at the start.
    """
    if n_available <= 0:
        return []
    if n_available <= n_want:
        return list(range(n_available))
    idxs = np.linspace(0, n_available - 1, n_want).round().astype(int)
    return sorted(set(idxs.tolist()))


def load_sequence_tensor(frames: List[dict], root: Path, category: str, seq_name: str,
                         transform, n_frames: int = 16,
                         device: str = "cpu") -> Dict[str, Any]:
    """Load one sequence into a padded tensor + validity mask.

    Returns dict with:
        frames  : [n_frames, 3, H, W] float32
        mask    : [n_frames] bool  (True = real frame)
        n_real  : number of genuine frames
    """
    from PIL import Image

    ordered = sorted(frames, key=lambda f: f.get("frame_number", 0))
    idxs = sample_frame_indices(len(ordered), n_frames)

    img_size = CONFIG["IMG_SIZE"]
    images = torch.zeros(n_frames, 3, img_size, img_size)
    mask = torch.zeros(n_frames, dtype=torch.bool)

    n_real = 0
    for slot, fi in enumerate(idxs):
        fr = ordered[fi]
        rel = (fr.get("image") or {}).get("path", "")
        p = resolve_image_path(root, category, seq_name, rel)
        if p is None:
            continue
        try:
            with Image.open(p) as im:
                im = im.convert("RGB")
                images[slot] = transform(im)
            mask[slot] = True
            n_real += 1
        except Exception:
            continue

    # Pad the tail by repeating the last valid frame (mask stays False for those slots).
    if n_real > 0 and n_real < n_frames:
        last_valid = int(torch.nonzero(mask)[-1].item())
        for slot in range(n_real, n_frames):
            images[slot] = images[last_valid]

    return {"frames": images, "mask": mask, "n_real": n_real}


if TORCH_OK:
    print("Preprocessing pipeline ready.")
    print(f"  Sample output: frames {[CONFIG['BATCH_SIZE'], CONFIG['NUM_FRAMES'], 3, CONFIG['IMG_SIZE'], CONFIG['IMG_SIZE']]}"
          f"  mask {[CONFIG['BATCH_SIZE'], CONFIG['NUM_FRAMES']]}")
else:
    print("[!] PyTorch missing — preprocessing cells are inert.")

Preprocessing pipeline ready.
  Sample output: frames [8, 16, 3, 224, 224]  mask [8, 16]


In [23]:
# ---------------------------------------------------------------------------------------------
# Dataset + DataLoader
# ---------------------------------------------------------------------------------------------

class SequenceQualityDataset(torch.utils.data.Dataset):
    """One item = one CO3D sequence.

    Returns frames [T,3,H,W], validity mask [T], quality score (scalar), label index,
    and the sequence id for error analysis.
    """

    LABELS = ["Poor", "Good"]     # index 0 = Poor, 1 = Good

    def __init__(self, records: List[dict], ann: Dict[str, Path],
                 root: Path, transform, n_frames: int = 16, n_want: int = 16):
        self.records = records
        self.ann = ann
        self.root = root
        self.transform = transform
        self.n_frames = n_frames
        self.n_want = n_want

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, i: int):
        rec = self.records[i]
        cat = rec["category"]
        seq = rec["sequence_name"]
        frames = self.ann[cat]["frames_by_sequence"].get(seq, [])
        out = load_sequence_tensor(frames, self.root, cat, seq, self.transform,
                                   n_frames=self.n_frames)
        score = float(rec.get("quality_score", 0.0))
        label = str(rec.get("label", "Poor"))
        y = torch.tensor(self.LABELS.index(label) if label in self.LABELS else 0,
                         dtype=torch.long)
        return {
            "frames": out["frames"],
            "mask": out["mask"],
            "score": torch.tensor(score, dtype=torch.float32),
            "label": y,
            "sequence_name": seq,
            "category": cat,
            "n_real": out["n_real"],
        }


def make_loaders(split_records: Dict[str, List[dict]], ann_by_cat: Dict[str, Any],
                 root: Path, batch_size: int = 8, n_frames: int = 16,
                 num_workers: int = 0) -> Dict[str, Any]:
    """Create train/val/test loaders. num_workers=0 is the safe default (Windows + notebooks)."""
    tf = build_transform(CONFIG["IMG_SIZE"])
    loaders = {}
    for split, recs in split_records.items():
        if not recs:
            continue
        ds = SequenceQualityDataset(recs, ann_by_cat, root, tf, n_frames=n_frames)
        loaders[split] = torch.utils.data.DataLoader(
            ds, batch_size=batch_size, shuffle=(split == "train"),
            num_workers=num_workers, drop_last=False)
    return loaders


# Build split records from the dataframe
split_records: Dict[str, List[dict]] = {"train": [], "val": [], "test": []}
if not LABEL_DF.empty:
    for _, r in LABEL_DF.iterrows():
        rec = r.to_dict()
        split_records.setdefault(rec.get("split", "train"), []).append(rec)

loaders = make_loaders(split_records, ANN, CONFIG["CO3D_ROOT"],
                        batch_size=CONFIG["BATCH_SIZE"], n_frames=CONFIG["NUM_FRAMES"])

if TORCH_OK and loaders:
    for split, dl in loaders.items():
        print(f"{split:<6} {len(dl.dataset):>4} sequences, {len(dl)} batches")
    # Show one batch's real shapes
    b = next(iter(loaders["train"]))
    print(f"\nBatch shapes: frames={tuple(b['frames'].shape)}  mask={tuple(b['mask'].shape)}  "
          f"score={tuple(b['score'].shape)}  label={tuple(b['label'].shape)}")
    print(f"Expected frames tensor: [{CONFIG['BATCH_SIZE']}, {CONFIG['NUM_FRAMES']}, 3, "
          f"{CONFIG['IMG_SIZE']}, {CONFIG['IMG_SIZE']}]")
else:
    print("[!] Cannot build loaders (no PyTorch or no labelled data).")

[!] Cannot build loaders (no PyTorch or no labelled data).


## 8. Feature extraction

Two feature families, both computable from RGB frames alone (no ground-truth geometry):

### 8.1 Spatial features — ResNet-18

A pretrained ResNet-18, with the classification head removed, maps each 224×224 frame to a
**512-dimensional** pooled feature vector. The ImageNet weights already encode useful priors for
sharpness, lighting, texture, and occlusion, so even a frozen encoder provides a meaningful signal.

> These are *learned representations*, not explicit measurements of sharpness or exposure. The
> model learns to map them to reconstruction quality; it does not "read off" a blur score.

### 8.2 Temporal features — computed from the RGB sequence

Cheap, deterministic statistics that describe how the video moves:

| Feature | Meaning | Why it matters for reconstruction |
|---|---|---|
| `mean_frame_diff` | Mean absolute pixel difference between consecutive frames | Low ⇒ static or near-duplicate views ⇒ poor parallax |
| `max_frame_diff` | Largest inter-frame change | Detects abrupt camera jumps / motion blur |
| `std_frame_diff` | Variability of inter-frame change | Erratic motion is harder to register |
| `n_near_duplicates` | Pairs with similarity above a threshold | Too many duplicates ⇒ insufficient viewpoint coverage |
| `viewpoint_diversity` | 1 − normalised mean pairwise feature distance | Directly proxies *baseline* — the thing SfM needs |
| `sharpness` | Mean variance of Laplacian | Blur ⇒ weak features ⇒ few correspondences |
| `exposure` | Mean luminance and its std | Under/over-exposure ⇒ few reliable keypoints |
| `mask_coverage` | Foreground fraction (from the *computed* mask, not GT) | Object must occupy enough pixels to be reconstructed |

### 8.3 The inference-time constraint

> **No ground-truth camera poses, depth maps, point clouds, or reconstruction metrics are ever
> model inputs.**

If the model consumed CO3D's GT poses it would be reading the answer key. If it consumed GT masks
it would require an expensive segmentation step on every new video. The mask-coverage feature is
therefore computed from a **simple, fast foreground estimator** (Otsu thresholding on a blurred
copy) that runs in milliseconds on any new video — see `estimate_foreground_mask` in Section 14.

### 8.4 Feature caching

Frame features are computed once and cached to disk (`.npy` per sequence). Re-training the LSTM then
takes seconds instead of minutes, and the CNN encoder is never re-run for the same data.

In [24]:
# ---------------------------------------------------------------------------------------------
# ResNet-18 frame encoder
# ---------------------------------------------------------------------------------------------

if TORCH_OK:
    from torchvision.models import resnet18, ResNet18_Weights

    def build_resnet18_encoder(pretrained: bool = True) -> nn.Module:
        """ResNet-18 with the ImageNet classification head removed -> 512-d pooled features."""
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        model = resnet18(weights=weights)
        model.fc = nn.Identity()          # 512-d output
        return model

    # Sanity check the feature dimension
    _enc = build_resnet18_encoder(pretrained=True).eval()
    with torch.no_grad():
        _f = _enc(torch.zeros(2, 3, CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"]))
    print(f"ResNet-18 encoder output shape: {tuple(_f.shape)}  "
          f"(expected [2, {CONFIG['EMBED_DIM']}])")
    del _enc
else:
    print("[!] PyTorch missing — encoder unavailable.")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 172MB/s]


ResNet-18 encoder output shape: (2, 512)  (expected [2, 512])


In [25]:
# ---------------------------------------------------------------------------------------------
# Temporal / image-quality features (RGB-only, inference-time computable)
# ---------------------------------------------------------------------------------------------

def estimate_foreground_mask(rgb: np.ndarray) -> np.ndarray:
    """Fast, dependency-light foreground estimate via Otsu thresholding.

    Used to derive the mask-coverage feature at inference time. This is NOT the CO3D
    ground-truth mask — it is a cheap approximation available for any new video.
    """
    from PIL import Image
    gray = np.asarray(Image.fromarray(rgb).convert("L"), dtype=np.float32)
    # Downscale for speed
    small = gray[::4, ::4]
    try:
        import cv2
        thr, _ = cv2.threshold(small.astype(np.uint8), 0, 255,
                                cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        mask = (gray >= thr)
    except Exception:
        # Pure-numpy Otsu fallback
        hist, _ = np.histogram(small, bins=256, range=(0, 255))
        total = small.size
        omega = np.cumsum(hist) / total
        mu = np.cumsum(hist * np.arange(256)) / total
        mu_t = mu[-1]
        denom = omega * (1 - omega)
        with np.errstate(divide="ignore", invalid="ignore"):
            sigma_b = np.where(denom > 0, (mu_t * omega - mu) ** 2 / denom, 0)
        thr = int(np.argmax(sigma_b))
        mask = (gray >= thr)
    return mask


def laplacian_variance(gray: np.ndarray) -> float:
    """Sharpness proxy: variance of the Laplacian. Higher = sharper."""
    k = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)
    try:
        from scipy.signal import convolve2d
        return float(np.var(convolve2d(gray, k, mode="same")))
    except Exception:
        # 4-neighbour discrete Laplacian fallback
        lap = (gray[:-1, 1:] + gray[1:, 1:] + gray[1:, :-1] + gray[:-1, :-1]
               - 4 * gray[1:-1, 1:-1])
        return float(np.var(lap))


def compute_temporal_features(frames: np.ndarray, feature_seq: np.ndarray) -> Dict[str, float]:
    """Compute motion / quality features for one sequence.

    Args:
        frames      : [T, H, W, 3] uint8 RGB, in temporal order
        feature_seq : [T, D] CNN feature vectors, aligned with frames
    Returns:
        dict of scalar features
    """
    T = len(frames)
    feats: Dict[str, float] = {}

    # --- inter-frame pixel difference ---
    if T >= 2:
        gray = [np.asarray(Image.fromarray(f).convert("L"), dtype=np.float32) for f in frames]
        diffs = np.array([np.mean(np.abs(gray[i + 1] - gray[i])) for i in range(T - 1)])
        feats["mean_frame_diff"] = float(diffs.mean())
        feats["max_frame_diff"] = float(diffs.max())
        feats["std_frame_diff"] = float(diffs.std())
        feats["n_near_duplicates"] = float((diffs < 2.0).sum())  # near-identical consecutive pairs
    else:
        feats.update({"mean_frame_diff": 0.0, "max_frame_diff": 0.0,
                      "std_frame_diff": 0.0, "n_near_duplicates": 0.0})

    # --- viewpoint diversity from CNN features ---
    if T >= 2 and feature_seq is not None and len(feature_seq) == T:
        f = feature_seq / (np.linalg.norm(feature_seq, axis=1, keepdims=True) + 1e-8)
        sim = f @ f.T
        n = T
        off = sim[~np.eye(n, dtype=bool)]
        feats["viewpoint_diversity"] = float(1.0 - np.mean(off)) if off.size else 0.0
    else:
        feats["viewpoint_diversity"] = 0.0

    # --- image quality ---
    grays = [np.asarray(Image.fromarray(f).convert("L"), dtype=np.float32) for f in frames]
    feats["sharpness"] = float(np.mean([laplacian_variance(g) for g in grays]))
    lum = np.array([g.mean() for g in grays])
    feats["exposure_mean"] = float(lum.mean())
    feats["exposure_std"] = float(lum.std())
    feats["underexposed_frac"] = float((lum < 40).mean())
    feats["overexposed_frac"] = float((lum > 220).mean())

    # --- foreground coverage (computed, not ground truth) ---
    covs = []
    for f in frames:
        m = estimate_foreground_mask(f)
        covs.append(float(m.mean()))
    feats["mask_coverage"] = float(np.mean(covs))
    feats["mask_coverage_std"] = float(np.std(covs))

    return feats


TEMPORAL_FEATURE_NAMES = [
    "mean_frame_diff", "max_frame_diff", "std_frame_diff", "n_near_duplicates",
    "viewpoint_diversity", "sharpness", "exposure_mean", "exposure_std",
    "underexposed_frac", "overexposed_frac", "mask_coverage", "mask_coverage_std",
]
print(f"Temporal feature set defined ({len(TEMPORAL_FEATURE_NAMES)} features):")
print("  " + ", ".join(TEMPORAL_FEATURE_NAMES))

Temporal feature set defined (12 features):
  mean_frame_diff, max_frame_diff, std_frame_diff, n_near_duplicates, viewpoint_diversity, sharpness, exposure_mean, exposure_std, underexposed_frac, overexposed_frac, mask_coverage, mask_coverage_std


In [26]:
# ---------------------------------------------------------------------------------------------
# Feature extraction + caching
# ---------------------------------------------------------------------------------------------

def extract_sequence_features(frames: List[dict], root: Path, category: str, seq_name: str,
                              encoder, transform, n_frames: int = 16,
                              device: str = "cpu") -> Dict[str, Any]:
    """Run the CNN over the sampled frames and compute temporal features.

    Returns {"cnn": [T, 512] float32, "temporal": {name: value}, "frame_paths": [...]}
    """
    from PIL import Image

    ordered = sorted(frames, key=lambda f: f.get("frame_number", 0))
    idxs = sample_frame_indices(len(ordered), n_frames)

    imgs, paths = [], []
    for fi in idxs:
        fr = ordered[fi]
        rel = (fr.get("image") or {}).get("path", "")
        p = resolve_image_path(root, category, seq_name, rel)
        if p is None:
            continue
        try:
            with Image.open(p) as im:
                imgs.append(np.asarray(im.convert("RGB"), dtype=np.uint8))
            paths.append(str(p))
        except Exception:
            continue

    if not imgs:
        return {"cnn": np.zeros((0, CONFIG["EMBED_DIM"]), dtype=np.float32),
                "temporal": {k: 0.0 for k in TEMPORAL_FEATURE_NAMES},
                "frame_paths": []}

    batch = torch.stack([transform(Image.fromarray(a)) for a in imgs]).to(device)
    encoder.eval()
    with torch.no_grad():
        cnn = encoder(batch).cpu().numpy().astype(np.float32)

    temporal = compute_temporal_features(np.stack(imgs), cnn)
    return {"cnn": cnn, "temporal": temporal, "frame_paths": paths}


def build_feature_cache(ann_by_cat: Dict[str, Any], records: List[dict], root: Path,
                        device: str = "cpu", force: bool = False) -> Dict[str, Any]:
    """Extract and cache features for all sequences in `records`."""
    if not TORCH_OK:
        raise RuntimeError("PyTorch required for feature extraction")

    encoder = build_resnet18_encoder(pretrained=True).to(device).eval()
    tf = build_transform(CONFIG["IMG_SIZE"])

    cache = {}
    for rec in records:
        cat, seq = rec["category"], rec["sequence_name"]
        key = f"{cat}__{seq}"
        fp = DIRS["features"] / f"{key}.npz"
        if fp.exists() and not force:
            with np.load(fp, allow_pickle=True) as z:
                cache[key] = {
                    "cnn": z["cnn"],
                    "temporal": {k: float(v) for k, v in zip(z["temporal_names"], z["temporal"])},
                }
            continue
        frames = ann_by_cat.get(cat, {}).get("frames_by_sequence", {}).get(seq, [])
        res = extract_sequence_features(frames, root, cat, seq, encoder, tf,
                                        n_frames=CONFIG["NUM_FRAMES"], device=device)
        np.savez_compressed(
            fp,
            cnn=res["cnn"],
            temporal=np.array([res["temporal"].get(k, 0.0) for k in TEMPORAL_FEATURE_NAMES],
                              dtype=np.float32),
            temporal_names=np.array(TEMPORAL_FEATURE_NAMES),
        )
        cache[key] = res
    return cache


if TORCH_OK and not LABEL_DF.empty:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Extracting features on {device} ...")
    FEATURE_CACHE = build_feature_cache(
        ANN, LABEL_DF.to_dict("records"), CONFIG["CO3D_ROOT"], device=device)
    print(f"Cached features for {len(FEATURE_CACHE)} sequences "
          f"in {DIRS['features']}")
else:
    FEATURE_CACHE = {}
    print("[!] Skipping feature extraction (no PyTorch or no labelled data).")

[!] Skipping feature extraction (no PyTorch or no labelled data).


## 9. CNN-LSTM architecture

```
RGB frames  [B, T, 3, 224, 224]
        │
        ▼
   ResNet-18 (shared weights, applied frame-by-frame)
        │      → [B, T, 512]
        ▼
   Temporal-feature MLP (optional side channel)
        │      → [B, T, 12]
        ▼
   Concatenate → [B, T, 524]
        │
        ▼
   LSTM (hidden 256, 1 layer) + packed-sequence masking
        │      → [B, 256]
        ▼
   Regression head: 256 → 128 → 1  (sigmoid × 100)
        │
        ▼
   Quality score ∈ [0, 100]   +   Good/Poor from the validated threshold
```

### 9.1 Why this architecture

1. **ResNet-18 for spatial features.** Small, fast, ImageNet-pretrained — and its 512-d pooled
   representation is a well-established general-purpose image descriptor. A student-scale project
   does not need a video transformer; the temporal task here is far simpler than action recognition.
2. **LSTM for temporal structure.** The quantity that decides reconstructability is largely
   *temporal*: how much the viewpoint moves, how much overlap consecutive frames share, whether the
   camera is erratic. An LSTM over ordered frame features captures exactly that, and unlike a
   temporal CNN it handles variable-length inputs natively via packing.
3. **A small regression head.** Two FC layers are enough to map a 256-d temporal state to a scalar.
   Keeping the head small limits overfitting, which matters when the labelled set is small.

### 9.2 The score is a *regression*, not a probability

The output is a **sigmoid-scaled scalar in [0, 100]**. It is a quality estimate, *not* a calibrated
probability that COLMAP will succeed. Do not present it as one. The Good/Poor label is a thresholded
version of that scalar, not an independent classifier head.

### 9.3 Freezing strategy

Training starts with the CNN **fully frozen** (only the LSTM and head learn). This:
- prevents a small dataset from destroying the pretrained representation,
- makes the first experiment fast and stable,
- and gives a clean baseline to compare partial fine-tuning against.

In [27]:
# ---------------------------------------------------------------------------------------------
# Model definition
# ---------------------------------------------------------------------------------------------

if TORCH_OK:

    class CNNLSTMQualityModel(nn.Module):
        """ResNet-18 (spatial) -> LSTM (temporal) -> regression head (score 0-100)."""

        def __init__(self, embed_dim: int = 512, lstm_hidden: int = 256, lstm_layers: int = 1,
                     dropout: float = 0.2, n_temporal: int = 12, use_temporal: bool = True,
                     pretrained: bool = True):
            super().__init__()
            self.use_temporal = use_temporal
            self.embed_dim = embed_dim

            # --- spatial encoder ---
            self.cnn = build_resnet18_encoder(pretrained=pretrained)

            # --- optional temporal-feature side channel ---
            self.temporal_mlp = nn.Sequential(
                nn.Linear(n_temporal, 64), nn.ReLU(), nn.Dropout(dropout)
            ) if use_temporal else None
            in_dim = embed_dim + (64 if use_temporal else 0)

            # --- temporal model ---
            self.lstm = nn.LSTM(input_size=in_dim, hidden_size=lstm_hidden,
                                num_layers=lstm_layers, batch_first=True,
                                dropout=dropout if lstm_layers > 1 else 0.0)

            # --- regression head ---
            self.head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(lstm_hidden, 128),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(128, 1),
            )

        def freeze_cnn(self, freeze: bool = True) -> None:
            for p in self.cnn.parameters():
                p.requires_grad = not freeze

        def forward(self, frames: torch.Tensor, mask: Optional[torch.Tensor] = None,
                    temporal: Optional[torch.Tensor] = None) -> torch.Tensor:
            """
            frames : [B, T, 3, H, W]
            mask   : [B, T] bool — True for real frames
            temporal: [B, T, F] optional
            returns: [B] quality score in [0, 100]
            """
            B, T = frames.shape[:2]

            # --- CNN over each frame (reshape to batch of frames) ---
            flat = frames.reshape(B * T, *frames.shape[2:])
            feats = self.cnn(flat)                        # [B*T, 512]
            feats = feats.reshape(B, T, -1)               # [B, T, 512]

            if self.use_temporal and temporal is not None:
                tfeat = self.temporal_mlp(temporal)      # [B, T, 64]
                feats = torch.cat([feats, tfeat], dim=-1) # [B, T, 512+64]

            # --- LSTM with packed sequences (ignores padded frames) ---
            lengths = mask.sum(dim=1).cpu().clamp(min=1) if mask is not None \
                else torch.full((B,), T, dtype=torch.long)

            packed = nn.utils.rnn.pack_padded_sequence(
                feats, lengths, batch_first=True, enforce_sorted=False)
            _, (h_n, _) = self.lstm(packed)
            temporal_state = h_n[-1]                     # [B, hidden]

            # --- regression head ---
            score = self.head(temporal_state).squeeze(-1)  # [B]
            return score * 100.0                          # sigmoid-bounded to [0,100]

    # Instantiate and print the architecture
    model = CNNLSTMQualityModel(
        embed_dim=CONFIG["EMBED_DIM"],
        lstm_hidden=CONFIG["LSTM_HIDDEN"],
        lstm_layers=CONFIG["LSTM_LAYERS"],
        dropout=CONFIG["DROPOUT"],
        n_temporal=len(TEMPORAL_FEATURE_NAMES),
        use_temporal=True,
    )
    model.freeze_cnn(True)

    n_params = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model parameters: {n_params:,} total, {n_train:,} trainable (CNN frozen)")
    print(model)

    # Forward-pass sanity check with the exact documented tensor shape
    with torch.no_grad():
        demo = torch.randn(2, CONFIG["NUM_FRAMES"], 3, CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"])
        dmask = torch.ones(2, CONFIG["NUM_FRAMES"], dtype=torch.bool)
        dtemp = torch.randn(2, CONFIG["NUM_FRAMES"], len(TEMPORAL_FEATURE_NAMES))
        out = model(demo, dmask, dtemp)
    print(f"\nForward check: input {tuple(demo.shape)} -> output {tuple(out.shape)}, "
          f"range [{out.min():.1f}, {out.max():.1f}]")
else:
    print("[!] PyTorch missing — model not defined.")

Model parameters: 12,064,385 total, 887,873 trainable (CNN frozen)
CNNLSTMQualityModel(
  (cnn): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
   

## 10. Model training and validation

### 10.1 Recipe

| Choice | Value | Why |
|---|---|---|
| Loss | **Huber** (`delta=10`) | Robust to outlier targets; MSE would let a few extreme scores dominate the gradient |
| Optimizer | **AdamW** | Decoupled weight decay generalises better than plain Adam on small noisy datasets |
| LR (CNN, frozen) | 1e-3 | CNN is not training, so only LSTM/head are affected |
| LR (CNN, fine-tune) | 1e-5 | 100× lower — enough to adapt, not enough to destroy pretrained weights |
| LR (head) | 1e-3 | Standard for a small MLP head |
| Batch size | 8 | Small; suits a small labelled set and modest VRAM |
| Epochs | ≤ 60 with early stopping (patience 10) | Stops before memorising a small dataset |
| Seeding | Fixed across numpy / torch / random | Reproducibility |

### 10.2 Two-stage training

1. **Stage A — frozen CNN.** Train LSTM + head only. This is the baseline.
2. **Stage B — partial fine-tuning.** Unfreeze the last ResNet block (`layer4`) at a low LR and
   continue training. Compare validation performance against Stage A and keep the better model.

### 10.3 Known extension point (not enabled by default)

Test-time augmentation (horizontal flip) and Gaussian blur augmentation are standard for small
video datasets. They are **not** enabled here by default because they change the effective
distribution of the sharpness feature, and the preprocessing must stay identical between
training and inference (Section 14). Enable them only if you also record the change in the saved
`preprocessing_config.json`.

In [28]:
# ---------------------------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------------------------

if TORCH_OK:

    def set_seed(seed: int) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    def temporal_tensor_for(record: dict, cache: Dict[str, Any], device: str) -> torch.Tensor:
        """Assemble the [T, F] temporal feature tensor for one record (with padding)."""
        key = f"{record['category']}__{record['sequence_name']}"
        entry = cache.get(key)
        T = CONFIG["NUM_FRAMES"]
        F = len(TEMPORAL_FEATURE_NAMES)
        if entry is None:
            return torch.zeros(T, F)
        vals = np.array([entry["temporal"].get(k, 0.0) for k in TEMPORAL_FEATURE_NAMES],
                        dtype=np.float32)
        n = len(vals)
        out = np.zeros((T, F), dtype=np.float32)
        out[:min(n, T)] = vals[:T]
        if n < T:                      # repeat last row for padded slots
            if n > 0:
                out[n:] = out[n - 1]
        return torch.from_numpy(out)

    def run_epoch(model, loader, criterion, optimizer=None, device="cpu",
                  temporal_cache=None, scaler=None):
        """One pass over the data. Trains if optimizer is given, else evaluates."""
        train_mode = optimizer is not None
        model.train(train_mode)
        device = next(model.parameters()).device

        total_loss, n_batches = 0.0, 0
        all_scores, all_targets, all_labels, all_names, all_cats = [], [], [], [], []

        for batch in loader:
            frames = batch["frames"].to(device, non_blocking=True)
            mask = batch["mask"].to(device, non_blocking=True)
            scores = batch["score"].to(device)
            labels = batch["label"].to(device)

            temporal = None
            if temporal_cache is not None and getattr(model, "use_temporal", False):
                temporal = torch.stack([
                    temporal_tensor_for({"category": c, "sequence_name": s},
                                        temporal_cache, device)
                    for c, s in zip(batch["category"], batch["sequence_name"])
                ]).to(device)

            with torch.set_grad_enabled(train_mode):
                pred = model(frames, mask, temporal)
                loss = criterion(pred, scores)

                if train_mode:
                    optimizer.zero_grad(set_to_none=True)
                    if scaler is not None:
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                        optimizer.step()

            total_loss += float(loss.item()) * frames.size(0)
            n_batches += frames.size(0)
            all_scores.append(pred.detach().cpu().numpy())
            all_targets.append(scores.detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())
            all_names.extend(list(batch["sequence_name"]))
            all_cats.extend(list(batch["category"]))

        return {
            "loss": total_loss / max(1, n_batches),
            "pred": np.concatenate(all_scores) if all_scores else np.array([]),
            "target": np.concatenate(all_targets) if all_targets else np.array([]),
            "label": np.concatenate(all_labels) if all_labels else np.array([]),
            "seq_names": all_names,
            "categories": all_cats,
        }

    def build_optimizer(model, lr_cnn: float, lr_head: float, weight_decay: float):
        """Two param groups: CNN (possibly fine-tuned, low LR) and the rest (higher LR)."""
        cnn_params = [p for p in model.cnn.parameters() if p.requires_grad]
        other_params = [p for p in model.parameters()
                        if p.requires_grad and not any(p is q for q in cnn_params)]
        groups = [{"params": other_params, "lr": lr_head, "weight_decay": weight_decay}]
        if cnn_params:
            groups.append({"params": cnn_params, "lr": lr_cnn, "weight_decay": weight_decay})
        return torch.optim.AdamW(groups, lr=lr_head, weight_decay=weight_decay)

    def train_model(model, loaders, temporal_cache, device="cpu", max_epochs=None,
                    lr_cnn=None, lr_head=None, patience=None, tag="stageA", verbose=True):
        """Train with early stopping on validation loss; restore the best weights at the end."""
        max_epochs = max_epochs or CONFIG["MAX_EPOCHS"]
        lr_cnn = lr_cnn if lr_cnn is not None else CONFIG["LR_CNN_FROZEN"]
        lr_head = lr_head if lr_head is not None else CONFIG["LR_HEAD"]
        patience = patience if patience is not None else CONFIG["EARLY_STOP_PATIENCE"]

        criterion = nn.HuberLoss(delta=10.0)
        optimizer = build_optimizer(model, lr_cnn, lr_head, CONFIG["WEIGHT_DECAY"])
        scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None

        best_val, best_state, bad_epochs = float("inf"), None, 0
        history = {"train_loss": [], "val_loss": [], "val_mae": [], "epoch": []}
        ckpt = DIRS["models"] / f"{tag}_best.pt"

        for epoch in range(1, max_epochs + 1):
            tr = run_epoch(model, loaders["train"], criterion, optimizer, device, temporal_cache, scaler)
            va = run_epoch(model, loaders["val"], criterion, None, device, temporal_cache, scaler) \
                if "val" in loaders else {"loss": float("nan"), "pred": np.array([]),
                                          "target": np.array([]), "label": np.array([]),
                                          "seq_names": [], "categories": []}
            val_mae = float(np.mean(np.abs(va["pred"] - va["target"]))) if len(va["pred"]) else float("nan")

            history["epoch"].append(epoch)
            history["train_loss"].append(tr["loss"])
            history["val_loss"].append(va["loss"])
            history["val_mae"].append(val_mae)

            if verbose:
                print(f"  epoch {epoch:>3}/{max_epochs}  train {tr['loss']:.4f}  "
                      f"val {va['loss']:.4f}  val_MAE {val_mae:.3f}")

            # Early stopping on validation loss
            if va["loss"] < best_val - 1e-5:
                best_val = va["loss"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                torch.save({"state_dict": best_state, "epoch": epoch, "val_loss": best_val}, ckpt)
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    if verbose:
                        print(f"  early stopping at epoch {epoch} (best val {best_val:.4f})")
                    break

        if best_state is not None:
            model.load_state_dict(best_state)
        return model, history, ckpt

    print("Training utilities ready.")
else:
    print("[!] PyTorch missing — training utilities not defined.")

Training utilities ready.


In [29]:
# ---------------------------------------------------------------------------------------------
# Stage A: train with the CNN frozen
# ---------------------------------------------------------------------------------------------

device = "cuda" if TORCH_OK and torch.cuda.is_available() else "cpu"

if TORCH_OK and loaders and "train" in loaders and "val" in loaders and len(loaders["train"].dataset) > 0:
    set_seed(SEED)
    model = CNNLSTMQualityModel(
        embed_dim=CONFIG["EMBED_DIM"], lstm_hidden=CONFIG["LSTM_HIDDEN"],
        lstm_layers=CONFIG["LSTM_LAYERS"], dropout=CONFIG["DROPOUT"],
        n_temporal=len(TEMPORAL_FEATURE_NAMES), use_temporal=True,
    ).to(device)
    model.freeze_cnn(True)

    print(f"STAGE A — frozen CNN.  device={device}  "
          f"train={len(loaders['train'].dataset)} val={len(loaders['val'].dataset)} sequences")
    model, hist_a, ckpt_a = train_model(
        model, loaders, FEATURE_CACHE, device=device,
        lr_cnn=CONFIG["LR_CNN_FROZEN"], lr_head=CONFIG["LR_HEAD"],
        max_epochs=CONFIG["MAX_EPOCHS"], patience=CONFIG["EARLY_STOP_PATIENCE"], tag="stageA")
    print(f"\nBest checkpoint: {ckpt_a}")
    FINAL_MODEL, FINAL_HISTORY, FINAL_TAG = model, hist_a, "stageA_frozen"
else:
    print("[!] Cannot train Stage A (missing PyTorch, loaders, or data).")
    FINAL_MODEL, FINAL_HISTORY, FINAL_TAG = None, None, None

[!] Cannot train Stage A (missing PyTorch, loaders, or data).


In [30]:
# ---------------------------------------------------------------------------------------------
# Training curves
# ---------------------------------------------------------------------------------------------

if FINAL_HISTORY:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ep = FINAL_HISTORY["epoch"]
    axes[0].plot(ep, FINAL_HISTORY["train_loss"], label="train", linewidth=1.8)
    axes[0].plot(ep, FINAL_HISTORY["val_loss"], label="validation", linewidth=1.8)
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("Huber loss")
    axes[0].set_title("Training / validation loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(ep, FINAL_HISTORY["val_mae"], color="crimson", linewidth=1.8)
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("MAE (score points)")
    axes[1].set_title("Validation MAE"); axes[1].grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(DIRS["figures"] / "training_curves.png", dpi=120)
    plt.show()
    print(f"Saved: {DIRS['figures'] / 'training_curves.png'}")
else:
    print("No training history to plot.")

No training history to plot.


In [31]:
# ---------------------------------------------------------------------------------------------
# Stage B: partial fine-tuning of layer4 — compared against Stage A
# ---------------------------------------------------------------------------------------------

if TORCH_OK and FINAL_MODEL is not None and "val" in loaders:
    import copy
    model_b = copy.deepcopy(FINAL_MODEL)

    # Unfreeze only the last ResNet block.
    for p in model_b.cnn.parameters():
        p.requires_grad = False
    for p in model_b.cnn.layer4.parameters():
        p.requires_grad = True

    n_tr = sum(p.numel() for p in model_b.parameters() if p.requires_grad)
    print(f"STAGE B — fine-tuning layer4.  trainable params: {n_tr:,}")

    model_b, hist_b, ckpt_b = train_model(
        model_b, loaders, FEATURE_CACHE, device=device,
        lr_cnn=CONFIG["LR_CNN_FINETUNE"], lr_head=CONFIG["LR_HEAD"] / 2,
        max_epochs=CONFIG["MAX_EPOCHS"], patience=CONFIG["EARLY_STOP_PATIENCE"], tag="stageB")

    val_a = min(hist_a["val_loss"]) if hist_a["val_loss"] else float("inf")
    val_b = min(hist_b["val_loss"]) if hist_b["val_loss"] else float("inf")
    print(f"\nBest validation loss — Stage A (frozen): {val_a:.4f}")
    print(f"Best validation loss — Stage B (layer4) : {val_b:.4f}")
    if val_b < val_a:
        print("-> Keeping Stage B (fine-tuning improved validation loss).")
        FINAL_MODEL, FINAL_HISTORY, FINAL_TAG = model_b, hist_b, "stageB_finetuned"
    else:
        print("-> Keeping Stage A (fine-tuning did not improve validation loss).")
        FINAL_MODEL, FINAL_HISTORY, FINAL_TAG = FINAL_MODEL, hist_a, "stageA_frozen"
else:
    print("[!] Skipping Stage B (no trained Stage A model).")

[!] Skipping Stage B (no trained Stage A model).


## 11. Evaluation and baseline comparisons

### 11.1 Metrics

**Regression (the primary task)**
- MAE — mean absolute error in score points
- RMSE — penalises large errors more heavily
- R² — variance explained (reported when meaningful, i.e. when the target has real spread)

**Classification (derived from the regression)**
- Accuracy, Precision, Recall, F1
- Confusion matrix
- ROC-AUC (computed by sweeping the threshold over the predicted score — not from a separate head)

### 11.2 Baselines

A deep model is only interesting if it beats simple alternatives:

1. **Mean predictor** — always predicts the training-set mean score.
2. **Handcrafted-feature ridge regression** — a regularised linear model on the 12 temporal/image-quality
   features. This is a *strong* baseline: it tests whether the CNN is actually contributing anything.
3. **Random forest on handcrafted features** — a nonlinear version of the same.

> If the CNN-LSTM does not beat the handcrafted ridge model, that is a finding worth reporting,
> not something to hide. On small labelled sets it frequently does not.

### 11.3 Protocol

The **test split is used exactly once**, at the end. It is never balanced, augmented, or resampled
to improve numbers.

In [32]:
# ---------------------------------------------------------------------------------------------
# Evaluation metrics + baselines
# ---------------------------------------------------------------------------------------------

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, mean_absolute_error,
                             mean_squared_error, r2_score, mean_absolute_percentage_error)
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """MAE / RMSE / R² (+ MAPE). R² is NaN when the target has no variance."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    out = {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 and np.var(y_true) > 1e-9 else float("nan"),
        "MAPE": float(mean_absolute_percentage_error(y_true, y_pred)) * 100 if np.all(np.abs(y_true) > 1e-6) else float("nan"),
    }
    return out


def classification_metrics(y_true_bin: np.ndarray, y_score: np.ndarray,
                            threshold: float) -> Dict[str, Any]:
    """Binary metrics derived from a continuous score by thresholding."""
    y_pred_bin = (y_score >= threshold).astype(int)
    out = {
        "accuracy": float(accuracy_score(y_true_bin, y_pred_bin)),
        "precision": float(precision_score(y_true_bin, y_pred_bin, zero_division=0)),
        "recall": float(recall_score(y_true_bin, y_pred_bin, zero_division=0)),
        "f1": float(f1_score(y_true_bin, y_pred_bin, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1]).tolist(),
    }
    if len(np.unique(y_true_bin)) > 1:
        try:
            out["roc_auc"] = float(roc_auc_score(y_true_bin, y_score))
        except Exception:
            out["roc_auc"] = float("nan")
    else:
        out["roc_auc"] = float("nan")
    return out


def handcrafted_matrix(records: List[dict], cache: Dict[str, Any]) -> np.ndarray:
    """Build the [N, F] matrix of handcrafted features for a list of records."""
    X = []
    for r in records:
        key = f"{r['category']}__{r['sequence_name']}"
        entry = cache.get(key, {})
        t = entry.get("temporal", {})
        X.append([float(t.get(k, 0.0)) for k in TEMPORAL_FEATURE_NAMES])
    return np.asarray(X, dtype=np.float32)


print("Evaluation utilities ready.")

Evaluation utilities ready.


In [33]:
# ---------------------------------------------------------------------------------------------
# Fit baselines on TRAIN, evaluate on TEST
# ---------------------------------------------------------------------------------------------

results: Dict[str, Any] = {}
test_records = split_records.get("test", [])
train_records = split_records.get("train", [])

if train_records and test_records and FEATURE_CACHE:
    y_train = np.array([float(r.get("quality_score", 0.0)) for r in train_records])
    y_test = np.array([float(r.get("quality_score", 0.0)) for r in test_records])

    X_train = handcrafted_matrix(train_records, FEATURE_CACHE)
    X_test = handcrafted_matrix(test_records, FEATURE_CACHE)

    # --- Baseline 1: mean predictor ---
    mean_pred = np.full_like(y_test, y_train.mean())
    results["Baseline: mean predictor"] = {
        "regression": regression_metrics(y_test, mean_pred),
        "classification": classification_metrics((y_test >= CONFIG["GOOD_SCORE_THRESHOLD"]).astype(int),
                                                 mean_pred, CONFIG["GOOD_SCORE_THRESHOLD"]),
    }

    # --- Baseline 2: ridge on handcrafted features ---
    ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 25)))
    ridge.fit(X_train, y_train)
    ridge_pred = ridge.predict(X_test)
    results["Baseline: ridge (handcrafted)"] = {
        "regression": regression_metrics(y_test, ridge_pred),
        "classification": classification_metrics((y_test >= CONFIG["GOOD_SCORE_THRESHOLD"]).astype(int),
                                                 ridge_pred, CONFIG["GOOD_SCORE_THRESHOLD"]),
    }

    # --- Baseline 3: random forest on handcrafted features ---
    rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=2,
                               random_state=SEED, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    results["Baseline: random forest (handcrafted)"] = {
        "regression": regression_metrics(y_test, rf_pred),
        "classification": classification_metrics((y_test >= CONFIG["GOOD_SCORE_THRESHOLD"]).astype(int),
                                                 rf_pred, CONFIG["GOOD_SCORE_THRESHOLD"]),
    }

    # --- The deep model ---
    if FINAL_MODEL is not None and "test" in loaders:
        criterion = nn.HuberLoss(delta=10.0)
        te = run_epoch(FINAL_MODEL, loaders["test"], criterion, None, device, FEATURE_CACHE, None)
        results["CNN-LSTM (ours)"] = {
            "regression": regression_metrics(te["target"], te["pred"]),
            "classification": classification_metrics(
                (te["target"] >= CONFIG["GOOD_SCORE_THRESHOLD"]).astype(int),
                te["pred"], CONFIG["GOOD_SCORE_THRESHOLD"]),
            "seq_names": te["seq_names"],
            "categories": te["categories"],
            "preds": te["pred"],
            "targets": te["target"],
        }
else:
    print("[!] Not enough data for evaluation (need non-empty train and test splits).")

# --- Print the comparison table ---
if results:
    print("=" * 86)
    print(f"TEST-SET RESULTS  (threshold = {CONFIG['GOOD_SCORE_THRESHOLD']:.0f})")
    print("=" * 86)
    print(f"{'model':<40} {'MAE':>7} {'RMSE':>7} {'R2':>7} {'Acc':>7} {'F1':>7} {'AUC':>7}")
    print("-" * 86)
    for name, r in results.items():
        reg, cls = r["regression"], r["classification"]
        auc = cls.get("roc_auc", float("nan"))
        print(f"{name:<40} {reg['MAE']:>7.2f} {reg['RMSE']:>7.2f} {reg['R2']:>7.3f} "
              f"{cls['accuracy']:>7.3f} {cls['f1']:>7.3f} {auc:>7.3f}")

    # Confusion matrix for the deep model
    if "CNN-LSTM (ours)" in results:
        cm = np.array(results["CNN-LSTM (ours)"]["classification"]["confusion_matrix"])
        fig, ax = plt.subplots(figsize=(5, 4.2))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["Poor", "Good"])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["Poor", "Good"])
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title("Confusion matrix (test set)")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                        fontsize=16, color="white" if cm[i, j] > cm.max() / 2 else "black")
        fig.colorbar(im, ax=ax)
        fig.tight_layout()
        fig.savefig(DIRS["figures"] / "confusion_matrix.png", dpi=120)
        plt.show()
else:
    print("No results to report.")

[!] Not enough data for evaluation (need non-empty train and test splits).
No results to report.


## 12. Error analysis

A single aggregate MAE hides the failure modes that matter. This section breaks the errors down
along the axes that actually explain reconstruction failure:

| Bucket | Definition | What it tells you |
|---|---|---|
| Blurry | `sharpness` below the dataset median | Does the model detect blur, or just memorise category style? |
| Poorly lit | `exposure_mean` low or `underexposed_frac` high | Does the model read lighting? |
| Insufficient coverage | `viewpoint_diversity` below the median | Can it detect a static, single-viewpoint video? |
| Repeated views | `n_near_duplicates` above the median | Can it detect near-duplicate frames? |
| Good images but failed geometry | `sharpness` high **and** COLMAP status `failed` | The hardest case — the video looks fine but is geometrically unusable |
| Per-category consistency | Error broken down by CO3D category | Is the model category-agnostic or overfitting to object style? |

> **A note on interpretability:** the buckets are defined using the *handcrafted* features, not the
> model's internal attention. This is a deliberate limitation — the CNN's 512-d representation is
> not directly readable as "blur" or "lighting". See Section 15.

In [34]:
# ---------------------------------------------------------------------------------------------
# Error analysis
# ---------------------------------------------------------------------------------------------

if results and "CNN-LSTM (ours)" in results:
    r = results["CNN-LSTM (ours)"]
    names = r.get("seq_names", [])
    cats = r.get("categories", [])
    preds = np.asarray(r.get("preds", []))
    targets = np.asarray(r.get("targets", []))

    if len(preds):
        err = preds - targets
        recs = []
        for nm, ct, p, t in zip(names, cats, preds, targets):
            key = f"{ct}__{nm}"
            tf = FEATURE_CACHE.get(key, {}).get("temporal", {})
            recs.append({
                "sequence_name": nm, "category": ct,
                "target": t, "pred": p, "error": p - t, "abs_error": abs(p - t),
                **{k: float(tf.get(k, 0.0)) for k in TEMPORAL_FEATURE_NAMES},
            })
        ERR = pd.DataFrame(recs)

        # Bucket definitions from the handcrafted features
        med_sharp = ERR["sharpness"].median()
        med_div = ERR["viewpoint_diversity"].median()
        med_dup = ERR["n_near_duplicates"].median()

        def bucket(rw) -> str:
            tags = []
            if rw["sharpness"] < med_sharp:
                tags.append("blurry")
            if rw["exposure_mean"] < 60 or rw["underexposed_frac"] > 0.2:
                tags.append("poorly_lit")
            if rw["viewpoint_diversity"] < med_div:
                tags.append("low_coverage")
            if rw["n_near_duplicates"] > med_dup:
                tags.append("repeated_views")
            if not tags:
                tags.append("none")
            return ",".join(tags)

        ERR["bucket"] = ERR.apply(bucket, axis=1)

        print("=" * 78)
        print("ERROR ANALYSIS — test set")
        print("=" * 78)
        print(f"Overall MAE: {ERR['abs_error'].mean():.2f}  |  "
              f"mean signed error: {ERR['error'].mean():+.2f}  "
              f"(systematic over/under-prediction)")
        print()
        bucket_stats = ERR.groupby("bucket").agg(
            n=("abs_error", "size"),
            MAE=("abs_error", "mean"),
            bias=("error", "mean"),
            mean_target=("target", "mean"),
            mean_pred=("pred", "mean"),
        ).sort_values("MAE", ascending=False)
        print(bucket_stats.to_string())

        # The hardest case: good-looking frames but failed reconstruction
        if not LABEL_DF.empty:
            status_map = LABEL_DF.set_index("sequence_name")["reconstruction_status"].to_dict()
            ERR["recon_status"] = ERR["sequence_name"].map(status_map)
            hard = ERR[(ERR["sharpness"] >= med_sharp) & (ERR["recon_status"] == "failed")]
            if len(hard):
                print(f"\n'Hard case' — good image quality but failed reconstruction: n={len(hard)}")
                print(f"  MAE {hard['abs_error'].mean():.2f}  "
                      f"mean target {hard['target'].mean():.1f}  "
                      f"mean pred {hard['pred'].mean():.1f}")
                print("  Interpretation: the model is learning image quality, not geometry. "
                      "This is the key limitation of the current feature design.")
        else:
            print("\n(no label table available for the 'good images but failed geometry' bucket)")

        # Per-category consistency
        print("\nPer-category performance:")
        cat_stats = ERR.groupby("category").agg(
            n=("abs_error", "size"), MAE=("abs_error", "mean"),
            mean_target=("target", "mean"), mean_pred=("pred", "mean"),
        )
        print(cat_stats.to_string())

        # Residuals plot
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        ax = axes[0]
        ax.scatter(targets, preds, s=40, alpha=0.8, edgecolor="black", linewidth=0.4)
        lims = [min(targets.min(), preds.min()) - 5, max(targets.max(), preds.max()) + 5]
        ax.plot(lims, lims, "r--", linewidth=1.2, label="perfect prediction")
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_xlabel("True quality score"); ax.set_ylabel("Predicted quality score")
        ax.set_title("Predicted vs true (test set)"); ax.legend(); ax.grid(alpha=0.3)

        ax = axes[1]
        ax.bar(bucket_stats.index, bucket_stats["MAE"], color="steelblue", edgecolor="black")
        ax.set_ylabel("MAE (score points)")
        ax.set_title("MAE by failure bucket")
        ax.tick_params(axis="x", rotation=30)
        fig.tight_layout()
        fig.savefig(DIRS["figures"] / "error_analysis.png", dpi=120)
        plt.show()

        ERR.to_csv(DIRS["labels"] / "test_errors.csv", index=False)
        print(f"\nSaved per-sequence errors -> {DIRS['labels'] / 'test_errors.csv'}")
else:
    print("[!] No test predictions available for error analysis.")

[!] No test predictions available for error analysis.


## 13. Saving the trained model

Everything needed for inference is written to one directory:

```
models/
├── cnn_lstm_quality.pt        # weights + architecture hyper-parameters
├── preprocessing_config.json  # IMG_SIZE, NUM_FRAMES, mean/std, sampling rule
├── feature_config.json        # temporal feature names + how they are computed
└── threshold.json             # validated Good/Poor threshold + calibration info
```

The point of this section: **Section 14 must be able to reconstruct the model from these files
alone**, with no access to the training data, no access to COLMAP, and no notebook state.

In [35]:
# ---------------------------------------------------------------------------------------------
# Save model + configs
# ---------------------------------------------------------------------------------------------

SAVED = {"ok": False}

if FINAL_MODEL is not None:
    ckpt_path = DIRS["models"] / "cnn_lstm_quality.pt"
    torch.save({
        "state_dict": {k: v.detach().cpu() for k, v in FINAL_MODEL.state_dict().items()},
        "config": {
            "embed_dim": CONFIG["EMBED_DIM"],
            "lstm_hidden": CONFIG["LSTM_HIDDEN"],
            "lstm_layers": CONFIG["LSTM_LAYERS"],
            "dropout": CONFIG["DROPOUT"],
            "n_temporal": len(TEMPORAL_FEATURE_NAMES),
            "use_temporal": True,
        },
        "tag": FINAL_TAG,
        "good_score_threshold": CONFIG["GOOD_SCORE_THRESHOLD"],
        "seed": SEED,
    }, ckpt_path)

    with open(DIRS["models"] / "preprocessing_config.json", "w", encoding="utf-8") as f:
        json.dump({
            "img_size": CONFIG["IMG_SIZE"],
            "num_frames": CONFIG["NUM_FRAMES"],
            "sampling": "even temporal spread over frame_number order; last-frame padding for short sequences",
            "imagenet_mean": IMAGENET_MEAN,
            "imagenet_std": IMAGENET_STD,
            "resize_interpolation": "bilinear",
        }, f, indent=2)

    with open(DIRS["models"] / "feature_config.json", "w", encoding="utf-8") as f:
        json.dump({
            "temporal_feature_names": TEMPORAL_FEATURE_NAMES,
            "cnn": "torchvision resnet18 (IMAGENET1K_V1), fc removed -> 512-d pooled features",
            "foreground_mask": "Otsu thresholding on a downscaled grayscale copy (inference-time approximation, not CO3D GT)",
            "normalization": {
                "temporal": "z-scored using training-split statistics (see fit_feature_scaler below)"
            },
        }, f, indent=2)

    with open(DIRS["models"] / "threshold.json", "w", encoding="utf-8") as f:
        json.dump({
            "good_score_threshold": CONFIG["GOOD_SCORE_THRESHOLD"],
            "calibrated_on": "validation split only",
            "note": "Applied to the continuous 0-100 regression score. Not a calibrated probability.",
        }, f, indent=2)

    print("Saved:")
    for p in sorted(DIRS["models"].iterdir()):
        print(f"  {p.name:<30} {p.stat().st_size/1024:8.1f} KB")
    SAVED = {"ok": True, "path": str(ckpt_path)}
else:
    print("[!] No trained model to save.")

[!] No trained model to save.


In [36]:
# ---------------------------------------------------------------------------------------------
# Fit + save the temporal-feature scaler (training statistics only)
# ---------------------------------------------------------------------------------------------

if train_records and FEATURE_CACHE:
    Xtr = handcrafted_matrix(train_records, FEATURE_CACHE)
    scaler = StandardScaler().fit(Xtr)      # TRAIN ONLY — never fit on val/test
    with open(DIRS["models"] / "temporal_scaler.npz", "wb") as f:
        np.savez(f, mean=scaler.mean_, scale=scaler.scale_, var=scaler.var_)
    print("Saved temporal-feature scaler (fit on training split only):")
    for name, mu, sd in zip(TEMPORAL_FEATURE_NAMES, scaler.mean_, scaler.scale_):
        print(f"  {name:<22} mean={mu:>12.4f}  std={sd:>12.4f}")
else:
    print("[!] No training records — scaler not fitted.")

[!] No training records — scaler not fitted.


## 14. Predicting quality for a new video

This is the deliverable: a function that takes **any video file**, samples frames, and returns a
score and a Good/Poor verdict — **without running COLMAP or any reconstruction**.

### 14.1 The inference path

```
video file (mp4/avi/mov/...) or directory of frames
        │
        ▼
   validate input
        │
        ▼
   sample NUM_FRAMES frames in temporal order
        │
        ▼
   apply the EXACT training-time preprocessing (224×224, ImageNet norm)
        │
        ▼
   ResNet-18 → 512-d per frame  +  12 handcrafted temporal features
        │
        ▼
   LSTM (packed) → 256-d temporal state → regression head
        │
        ▼
   score ∈ [0, 100]  →  Good/Poor via the saved threshold
```

### 14.2 Important properties

- **No reconstruction.** The whole point. No COLMAP, no depth, no pose estimation.
- **Exact preprocessing match.** The saved `preprocessing_config.json` drives the transform, so
  training and inference cannot drift apart.
- **Graceful degradation.** A video with fewer frames than `NUM_FRAMES` is padded; a video that
  cannot be decoded raises a clear error with a suggested fix.
- **Honest output.** The returned dict includes a `limitations` field stating that the score is a
  quality estimate from a learned proxy, not a probability of reconstruction success.

In [37]:
# ---------------------------------------------------------------------------------------------
# Video reading (OpenCV) + the reusable inference function
# ---------------------------------------------------------------------------------------------

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v", ".mpg", ".mpeg"}


def read_video_frames(path: Path, max_frames: int = 512) -> List[np.ndarray]:
    """Read a video into a list of RGB uint8 frames.

    Reads at most `max_frames` frames to bound memory. Raises a clear error if the
    video cannot be opened or contains no decodable frames.
    """
    import cv2
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(
            f"Could not open video: {path}\n"
            "Check that the path is correct and that OpenCV has the codec (try installing "
            "`opencv-python` with ffmpeg support, or convert to a sequence of JPEGs).")
    frames = []
    while len(frames) < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames:
        raise RuntimeError(f"No frames could be decoded from: {path}")
    return frames


def read_frame_directory(path: Path) -> List[np.ndarray]:
    """Read a directory of images as a frame sequence (sorted by filename)."""
    from PIL import Image
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    files = sorted([p for p in Path(path).iterdir() if p.suffix.lower() in exts])
    if not files:
        raise RuntimeError(f"No image files found in: {path}")
    frames = []
    for p in files:
        try:
            with Image.open(p) as im:
                frames.append(np.asarray(im.convert("RGB"), dtype=np.uint8))
        except Exception:
            continue
    if not frames:
        raise RuntimeError(f"No readable images in: {path}")
    return frames


print("Video readers ready.")

Video readers ready.


In [38]:
# ---------------------------------------------------------------------------------------------
# The reusable prediction function
# ---------------------------------------------------------------------------------------------

def predict_video(video_path: str,
                  model_dir: str = None,
                  device: str = None,
                  return_explanation: bool = True) -> Dict[str, Any]:
    """Predict the reconstruction quality of a video.

    Args:
        video_path        : path to a video file, or a directory of frames
        model_dir         : directory holding the saved model + configs
                            (default: the project's models directory)
        device            : 'cuda' / 'cpu' (auto-detected by default)
        return_explanation: include a human-readable explanation and limitations

    Returns:
        dict with:
            score          : float in [0, 100]
            label          : 'Good' or 'Poor'
            threshold      : the threshold that was applied
            n_frames_used  : number of frames actually fed to the model
            n_frames_video : number of frames in the source
            features       : the handcrafted temporal features
            explanation    : str (optional)
            limitations    : str
    """
    if not TORCH_OK:
        raise RuntimeError("PyTorch is required for inference.")

    model_dir = Path(model_dir) if model_dir else DIRS["models"]
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # ---- 1. validate the input ----
    p = Path(video_path)
    if not p.exists():
        raise FileNotFoundError(f"Input not found: {video_path}")
    if p.is_dir():
        frames = read_frame_directory(p)
    elif p.suffix.lower() in VIDEO_EXTENSIONS:
        frames = read_video_frames(p)
    else:
        # Unknown extension: try as a video, then as a directory of images.
        try:
            frames = read_video_frames(p)
        except Exception:
            frames = read_frame_directory(p)

    if len(frames) < 1:
        raise RuntimeError("Input contained no usable frames.")

    # ---- 2. load the saved artefacts ----
    with open(model_dir / "preprocessing_config.json", "r", encoding="utf-8") as f:
        prep = json.load(f)
    with open(model_dir / "feature_config.json", "r", encoding="utf-8") as f:
        feat_cfg = json.load(f)
    with open(model_dir / "threshold.json", "r", encoding="utf-8") as f:
        thr_cfg = json.load(f)

    ckpt = torch.load(model_dir / "cnn_lstm_quality.pt", map_location="cpu",
                      weights_only=False)
    model = CNNLSTMQualityModel(pretrained=False, **ckpt["config"])
    model.load_state_dict(ckpt["state_dict"])
    model.to(device).eval()

    scaler_path = model_dir / "temporal_scaler.npz"
    scaler = None
    if scaler_path.exists():
        with np.load(scaler_path) as z:
            scaler = (z["mean"], z["scale"])

    # ---- 3. sample frames (EXACT training-time rule) ----
    n_want = prep["num_frames"]
    img_size = prep["img_size"]
    idxs = sample_frame_indices(len(frames), n_want)
    sampled = [frames[i] for i in idxs]

    tf = transforms.Compose([
        transforms.Resize((img_size, img_size),
                          interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.ToTensor(),
        transforms.Normalize(mean=prep["imagenet_mean"], std=prep["imagenet_std"]),
    ])

    # ---- 4. CNN features + handcrafted temporal features ----
    batch = torch.stack([tf(Image.fromarray(a)) for a in sampled]).unsqueeze(0).to(device)
    with torch.no_grad():
        cnn_feats = model.cnn(batch.reshape(n_want, 3, img_size, img_size)
                               .to(device)).cpu().numpy().astype(np.float32)
    temporal = compute_temporal_features(np.stack(sampled), cnn_feats)

    # ---- 5. LSTM + regression head ----
    F = len(feat_cfg["temporal_feature_names"])
    tvec = np.array([temporal.get(k, 0.0) for k in feat_cfg["temporal_feature_names"]],
                     dtype=np.float32)
    if scaler is not None:
        mu, sd = scaler
        tvec = (tvec - mu) / np.where(sd < 1e-8, 1.0, sd)
    # Repeat the single vector across the time axis to match the training layout
    t_tensor = torch.from_numpy(np.repeat(tvec[None, :], n_want, axis=0)).unsqueeze(0).to(device)

    mask = torch.ones(1, n_want, dtype=torch.bool, device=device)
    with torch.no_grad():
        score = float(model(batch, mask, t_tensor).item())

    score = float(np.clip(score, 0.0, 100.0))
    threshold = float(thr_cfg["good_score_threshold"])
    label = "Good" if score >= threshold else "Poor"

    out: Dict[str, Any] = {
        "score": round(score, 2),
        "label": label,
        "threshold": threshold,
        "n_frames_video": len(frames),
        "n_frames_used": len(sampled),
        "features": temporal,
    }

    if return_explanation:
        out["explanation"] = (
            f"Predicted reconstruction-quality score {score:.1f}/100 "
            f"({'at or above' if label == 'Good' else 'below'} the validated threshold "
            f"of {threshold:.0f}) -> {label}.\n"
            f"Evidence: mean inter-frame change {temporal.get('mean_frame_diff', 0):.2f}, "
            f"viewpoint diversity {temporal.get('viewpoint_diversity', 0):.3f}, "
            f"sharpness {temporal.get('sharpness', 0):.1f}, "
            f"foreground coverage {temporal.get('mask_coverage', 0):.3f}, "
            f"{temporal.get('n_near_duplicates', 0):.0f} near-duplicate frame pairs."
        )
        out["limitations"] = (
            "This is a learned quality estimate, NOT a calibrated probability that 3D "
            "reconstruction will succeed. The model was trained on CO3D object videos "
            "labelled by COLMAP reconstruction metrics, so it transfers best to similar "
            "capture conditions (moving camera around a mostly-static object) and may be "
            "unreliable for fast motion, full scenes, or unseen capture setups."
        )
    return out

In [39]:
# ---------------------------------------------------------------------------------------------
# Run the inference pipeline on a test sequence (sanity check) and on a user video
# ---------------------------------------------------------------------------------------------

def sequence_to_video(record: dict, root: Path, out_path: Path) -> Optional[Path]:
    """Write a CO3D sequence's sampled frames to a temp dir so predict_video can consume it."""
    cat, seq = record["category"], record["sequence_name"]
    frames = ANN.get(cat, {}).get("frames_by_sequence", {}).get(seq, [])
    if not frames:
        return None
    out_path.mkdir(parents=True, exist_ok=True)
    for f in out_path.glob("*.jpg"):
        f.unlink()
    ordered = sorted(frames, key=lambda f: f.get("frame_number", 0))
    for i, fr in enumerate(ordered):
        rel = (fr.get("image") or {}).get("path", "")
        src = resolve_image_path(root, cat, seq, rel)
        if src is None:
            continue
        try:
            shutil.copyfile(src, out_path / f"frame{i:05d}.jpg")
        except Exception:
            continue
    return out_path


if FINAL_MODEL is not None and SAVED.get("ok"):
    # --- sanity check on a known test sequence ---
    demo_dir = DIRS["features"] / "_demo_frames"
    demo_rec = (split_records.get("test") or split_records.get("val") or [None])[0]
    if demo_rec:
        sequence_to_video(demo_rec, CONFIG["CO3D_ROOT"], demo_dir)
        pred = predict_video(str(demo_dir))
        true_score = float(demo_rec.get("quality_score", float("nan")))
        print("=" * 78)
        print("SANITY CHECK — predicting on a known CO3D sequence")
        print("=" * 78)
        print(f"Sequence        : {demo_rec['category']}/{demo_rec['sequence_name']}")
        print(f"Ground-truth score: {true_score:.2f}  (label {demo_rec.get('label')})")
        print(f"Predicted score : {pred['score']:.2f}  (label {pred['label']})")
        print(f"Error           : {pred['score'] - true_score:+.2f}")
        print(f"\n{pred['explanation']}")
        print(f"\nLimitations: {pred['limitations']}")
    else:
        print("No demo sequence available.")

    # --- usage on a user's own video ---
    print("\n" + "=" * 78)
    print("TO PREDICT ON YOUR OWN VIDEO")
    print("=" * 78)
    print("""
    from IPython.display import FileLink  # optional
    result = predict_video("/path/to/your/video.mp4")
    print(result["score"], result["label"])
    print(result["explanation"])
    """)
else:
    print("[!] No saved model — inference function is defined but not executable yet.")

[!] No saved model — inference function is defined but not executable yet.


## 15. Final conclusions and limitations

### 15.1 What was built

A complete, reproducible pipeline that:

1. **Inspects** a real subset of CO3D — categories, sequences, frame counts, image sizes and
   formats, missing/corrupt files, and the availability of masks, depths, camera annotations and
   point clouds, all verified from disk rather than assumed.
2. **Constructs ground truth** by running real COLMAP SfM reconstructions and harvesting
   registration ratio, reprojection error, point-cloud density and surface coverage — then
   combining them into a documented 0–100 score with explicit weights and an explicit failure rule.
3. **Preprocesses** at the sequence level: temporal-order-preserving frame sampling, 224×224
   resize, ImageNet normalisation, and last-frame padding with a validity mask that the LSTM
   actually respects.
4. **Extracts features** from RGB alone — 512-d ResNet-18 spatial features plus 12 cheap temporal
   and image-quality features — with **no ground-truth pose, depth, point cloud or reconstruction
   metric entering the model**.
5. **Trains** a ResNet-18 + LSTM + regression head with Huber loss, AdamW, early stopping and
   checkpointing, first with the CNN frozen and then with partial fine-tuning of `layer4`.
6. **Evaluates** against three baselines, then breaks the errors down by failure mode and category.
7. **Deploys** a `predict_video()` function that returns a score, a Good/Poor verdict and an
   honest limitations statement — without running any reconstruction.

### 15.2 Honest limitations

**Of the labels**
- The score measures *COLMAP's success on these frames*, not absolute 3D accuracy.
- Scores are tied to the COLMAP version and feature/matcher settings; changing them shifts the labels.
- Coverage is unavailable for some sequences, so the effective metric mix varies slightly.
- The labelled set is small (dozens–hundreds), which is a small-sample regression problem.
- Sequences that fail to load are excluded entirely, biasing the label distribution towards
  processable data. All exclusions are recorded in `labels/exclusions.csv`.

**Of the model**
- The score is **not** a calibrated probability of reconstruction success.
- The handcrafted-feature baselines may match or beat the CNN-LSTM on a small labelled set. If
  they do, that is the honest result to report.
- The "good images but failed geometry" bucket is the model's real blind spot: the current input
  representation (RGB appearance + frame-to-frame change) does not directly encode the geometric
  properties that make SfM succeed, such as parallax magnitude or feature-track geometry.
- The ResNet-18 representation is not interpretable as "blur" or "lighting"; the error buckets are
  defined with handcrafted features, not the model's internals.
- Single-object, turntable-style capture is the training distribution. Transfer to full scenes,
  fast motion, or unseen capture rigs is unvalidated.

**Of the inference path**
- The foreground mask is an Otsu approximation, not a learned segmenter, so the mask-coverage
  feature is noisy on cluttered backgrounds.
- Sampling 16 frames from a long video may miss short critical segments.
- No confidence interval is reported; a calibrated uncertainty estimate would need quantile
  regression or an ensemble, neither of which was validated here.

### 15.3 Next steps, in priority order

1. **Close the geometric blind spot.** Add an explicit parallax estimate — optical-flow magnitude
   or feature-track displacement between distant frames — and a *learned* correspondence-quality
   score. These are computable at inference time and are the properties SfM actually depends on.
2. **Scale the labels.** Run COLMAP over a few hundred sequences across more categories. The
   pipeline is already resumable and cached.
3. **Multi-task learning.** Train the model to predict the *individual* COLMAP metrics
   (registration ratio, reprojection error, point density) as auxiliary heads, with the composite
   score as the final target. This gives more supervision signal and a more interpretable model.
4. **Better temporal modelling.** Compare the LSTM against a temporal-convolution encoder and a
   small transformer, on the same features, to see whether the temporal model is the bottleneck.
5. **Calibrated uncertainty.** Add a quantile-regression head or a deep ensemble, and validate
   whether the intervals are actually calibrated.
6. **Domain generalisation.** Evaluate on a dataset outside CO3D (e.g. ScanNet, Matterport3D, or
   self-captured phone video) to quantify the transfer gap.

### 15.4 Reproducibility

- Fixed seeds across `random`, `numpy` and `torch`.
- `cudnn.deterministic = True`, `cudnn.benchmark = False`.
- All intermediate artefacts cached under `work/` — deleting it forces a clean re-run.
- Every configuration constant lives in the `CONFIG` dict at the top of Section 2.
- The saved `models/` directory is sufficient for inference; no training data, no COLMAP, and no
  notebook state are required.

In [40]:
# ---------------------------------------------------------------------------------------------
# Final artifact summary
# ---------------------------------------------------------------------------------------------

print("=" * 78)
print("PROJECT ARTIFACTS")
print("=" * 78)

for label, folder in [("Labels", DIRS["labels"]), ("Features", DIRS["features"]),
                       ("Models", DIRS["models"]), ("Figures", DIRS["figures"]),
                       ("COLMAP workspaces", DIRS["colmap"])]:
    if folder.exists():
        files = [f for f in folder.rglob("*") if f.is_file()]
        size = sum(f.stat().st_size for f in files)
        print(f"  {label:<20} {len(files):>5} files  {size/1024/1024:8.2f} MB   {folder}")

print()
print("Key outputs to report:")
if "CNN-LSTM (ours)" in results:
    r = results["CNN-LSTM (ours)"]
    print(f"  • Test MAE              : {r['regression']['MAE']:.2f} score points")
    print(f"  • Test RMSE             : {r['regression']['RMSE']:.2f}")
    print(f"  • Test R²               : {r['regression']['R2']:.3f}")
    print(f"  • Test accuracy         : {r['classification']['accuracy']:.3f}")
    print(f"  • Test F1               : {r['classification']['f1']:.3f}")
    print(f"  • Good/Poor threshold   : {CONFIG['GOOD_SCORE_THRESHOLD']:.0f}")
    print(f"  • Trained variant       : {FINAL_TAG}")
else:
    print("  (no test results — the pipeline has not been run end-to-end yet)")
print("=" * 78)

PROJECT ARTIFACTS
  Labels                   0 files      0.00 MB   /content/work/labels
  Features                 0 files      0.00 MB   /content/work/features
  Models                   0 files      0.00 MB   /content/work/models
  Figures                  0 files      0.00 MB   /content/work/figures
  COLMAP workspaces        0 files      0.00 MB   /content/work/colmap

Key outputs to report:
  (no test results — the pipeline has not been run end-to-end yet)
